# Can any scorer separate variants that improved binding from variants that worsened it?

Every calibration in this repo so far has asked a scorer to **rank**: Spearman
against ddG in `03_adapt_calibration.ipynb`, Spearman against KD within a fixed
substitution count in `03_chinery_size_calibration.ipynb`. Ranking is the harder
question, both notebooks answer it badly, and that leaves the easier question
unanswered and worth asking on its own:

**given one variant measured to bind better and one measured to bind worse, does
the scorer put them in the right order?**

That is a question about direction, not magnitude, and it has one natural
statistic. AUROC *is* the probability that a randomly drawn positive outranks a
randomly drawn negative, so it is the question written as a number rather than a
proxy for it. A scorer at 0.5 is a coin. A scorer that cannot clear 0.5 on
direction has no business filtering a shortlist, whatever its correlation looks
like.

The reason this had not been asked is that the evidence set had almost no
negatives. `03_adapt_calibration.ipynb` leans on two - `L:S50F` and `L:S50W` -
and two negatives among eleven singles put a floor of **p = 1/C(11,2) = 0.018**
under any permutation test, which Part 2 reports so that what the extra
negatives bought is a number and not a feeling.
`data/benchmarks/trastuzumab_negatives.csv` now carries eight measured affinity
losses from the literature and several hundred labelled Chinery wells, which is
what makes this notebook possible.

| Part | What it does | Reads | Writes |
|---|---|---|---|
| 0 | Preflight: which scorers have a route on this box | `routes.py` | nothing |
| 1 | The labelled direction set: 21 positives, 8 negatives, deduplicated | both benchmark CSVs | `OUT_DIR/direction_set.csv` |
| 2 | Direction AUROC per scorer with a bootstrap CI - overall, singles against multi, by format | the structure, the scorers | `OUT_DIR/direction` |
| 3 | The Chinery binary: `positive` against `negative_clean`, overall and within each substitution count | the negatives CSV | `OUT_DIR/chinery` |
| 4 | Manufactured negatives as a floor check: hotspot alanine, implausible substitutions, non-interface decoys, `wt_argmax_rate` | the structure | `OUT_DIR/manufactured` |
| 5 | The verdict table, one row per scorer | Parts 2 to 4 | `OUT_DIR/verdict.csv` |

Parts 0 and 1 and every table in Parts 2 to 5 are cheap. The scoring cells need
Env B, a FoldX licence or a container, and Part 3 is the expensive one: 253
candidates for the clean split and 687 once the caveated negatives are in.

## What this notebook is not

**It is not a magnitude calibration.** Nothing here reports how much better a
positive was. Part 1 says why at length: three formats and three assays, and
only the sign survives pooling.

**It is not a replacement for the ranking notebooks.** A scorer can pass
direction and still rank badly inside the improved set, which is the question
`03_chinery_size_calibration.ipynb` asks and answers in the negative past four
substitutions. Direction is the weaker claim. It is here because the weaker
claim had never been checked, and because a scorer that fails it fails
everything downstream of it.

**It is not powered.** Twenty-one against eight is a confidence interval roughly
0.4 wide, so the interval is reported everywhere below and the point estimate
nowhere on its own. Several scorers are tested against the same small panels, so
Part 5 states how many were tested beside whichever one wins.

## Where the negatives come from, and what "positive" means in each part

Real negatives are scarce: papers report the variants that worked. So this
notebook assembles negatives from three different sources, and **the
positive/negative axis is not the same question in all of them**. Read this
before any AUROC below, because the label decides what the number licenses.

| Part | The negatives are | A "positive" is | Trust |
|---|---|---|---|
| **2 - direction set** | 8 variants measured as *worse* than the parent | one measured as *better* (21) | highest label quality, smallest n |
| **3 - Chinery binary** | 97 designs that bound but could not be fitted (plus 434 blanks, off by default) | a design the instrument returned a clean fitted KD for (156) | weaker label, real n |
| **4 - manufactured** | designs built to be bad on purpose: charge reversals, glycine for a buried bulky side chain, proline in a strand, alanine at a FoldX hotspot | surface decoys, expected neutral | unlimited n, but they encode our own assumptions |

So a scorer that separates in Part 2 is good at *affinity direction*; one that
separates in Part 3 is good at *telling a measurable binder from a weak one*;
one that only separates in Part 4 may be agreeing with the chemistry the decoys
were built from. The verdict table in Part 5 keeps the three apart for that
reason, and nothing here pools them.

### The negatives file counts substitutions, not designs

`data/benchmarks/trastuzumab_negatives.csv` has **4,551 rows but 700 designs**:
one row per substitution, a median of 7 per design and up to 10. Every count in
this notebook is designs, so `label_class` row counts will look ~7x larger than
the numbers below. By design: 434 `negative_caveated`, 156 `positive`, 97
`negative_clean` from `chinery2026`, plus 13 from `adapt2017` and `literature`
that supply Part 2's negatives.

### What the blanks are, and why they are off by default

Chinery's 434 `negative_caveated` are `N/A` wells. A blank is consistent with a
design that does not bind, with one that never expressed, and with a sensor that
did not regenerate - the source does not distinguish them. Scoring them costs
63% of this notebook's FoldX time to add labels that may be wrong in an unknown
direction, so `SCORE_CAVEATED` is **False**: Part 3 reports the clean split of
156 against 97 and says the second table was not computed, rather than
publishing a number built on ambiguous labels. Set it True only if you want that
second table and have the FoldX hours.

In [1]:
# --- parameters -------------------------------------------------------------

EVIDENCE = "data/benchmarks/trastuzumab_mutation_evidence.csv"
NEGATIVES = "data/benchmarks/trastuzumab_negatives.csv"

# --- Part 1: the labelled direction set -------------------------------------

# Positives. The ADAPT panel's improvers, selected by their own measurement
# rather than by a hand-written list: the 2017 Fab study, the metric that
# carries a fold change, and a fold change above 1. On the file as of
# 2026-11-03 that is 20 of the 22 ADAPT variants; the other two are L:S50F and
# L:S50W, which are negatives and arrive from the other file.
ADAPT_STUDY = "adapt2017"
ADAPT_METRIC = "affinity_fold_improvement"
ADAPT_IMPROVED_ABOVE = 1.0

# Moon 2016's D98W, the one improver outside ADAPT with a clean matched parent
# on the same run (0.11 nM against 0.48 nM). Named by evidence id, because "the
# moon2016 row with a KD" is seven variants and a control.
MOON_POSITIVE = "TZM-019"

EXPECTED_POSITIVES = 21
EXPECTED_POSITIVE_SIZES = {1: 10, 2: 7, 3: 4}

# Negatives. Selected by label, not by block: `negative_affinity` is the measured
# affinity losses, and it appears in both blocks of the negatives file - 8
# variants in the literature block and 2 more in the chinery block, carried over
# from ADAPT with no evidence id, which is the duplication the dedup step is
# about. `negative_developability` (heterogeneous on purification, too low to
# characterise) and `negative_function` (reduced ADCC, reduced cell binding) are
# real failures and are not a binding direction, so they are dropped and named.
# NEGATIVE_SOURCE is the literature block, which is where that report comes from.
NEGATIVE_LABEL = "negative_affinity"
NEGATIVE_SOURCE = "clearly_negative_variants"

EXPECTED_NEGATIVES = 8
EXPECTED_NEGATIVE_SIZES = {1: 3, 2: 2, 3: 2, 7: 1}

# L:S50F and L:S50W are in both files, and twice in the negatives file: once in
# the literature block with an evidence id, once in the chinery block carried
# over from ADAPT with none. The assertion in Part 1 is that deduplication finds
# exactly these two collisions - a third would mean the files have drifted.
EXPECTED_DUPLICATES = ("L:S50F", "L:S50W")

# The weak negative, named as a parameter so that the sensitivity check in
# Part 2 is a stated prediction and not a sentence written afterwards. The
# source calls the affinity "similar": 4.8 +/- 0.7 nM against 2.0 +/- 0.4 nM,
# no significance test.
WEAK_NEGATIVE = "L:R66G"

# --- the structure ----------------------------------------------------------

# The local 1N8Z: trastuzumab Fab bound to HER2 domain IV. The heavy chain is
# 'B', not 'H', and both benchmark files write H and L, so CHAIN_MAP is the one
# place that is translated and Part 2 asserts it against the coordinates.
PDB_PATH = None                    # None -> the candidates searched in Part 2
HEAVY_CHAIN = "B"
LIGHT_CHAIN = "A"
ANTIGEN_CHAIN = "C"
CHAIN_MAP = {"H": HEAVY_CHAIN, "L": LIGHT_CHAIN}

OUT_DIR = "runs/negative_controls"

# --- the scorers ------------------------------------------------------------

# Split by what each can answer, because the split decides which negatives it is
# graded on. A matrix scorer answers one substitution from a per-position lookup
# and returns an empty cell for a candidate with several - never a sum - so it
# sees 10 of the 21 positives and 3 of the 8 negatives. The multi-capable
# scorers see all of both.
MATRIX_MODELS = ("antifold", "ablang2", "esm1v", "esm2_650m", "antiberty", "consensus")
MULTI_MODELS = ("antifold_seq", "foldx", "gearbind")

# Env B's interpreter, the one that can `import antifold`. None falls back to
# $DESIGN_PIPELINE_ANTIFOLD, which is what the adapter reads anyway.
ANTIFOLD_PYTHON = None
ANTIFOLD_SRC = None                # None -> $ANTIFOLD_SRC; the matrix route needs it
ANTIFOLD_WEIGHTS = None            # a path, to reuse the 567 MB download
ANTIFOLD_DEVICE = "cuda"   # its environment has cu130 torch and sees the card

# FoldX comes out of the environment: FOLDX_BINARY, and DESIGN_PIPELINE_FOLDX_PDB
# pointing at the repaired, trimmed Fv plus antigen domain the alanine scan ran
# on (its residue numbers must still be 1N8Z's).
FOLDX_PYTHON = None                # None -> $DESIGN_PIPELINE_FOLDX
FOLDX_WORKERS = None
FOLDX_RUNS = None

GEARBIND_IMAGE = None              # no published image; None -> reported unavailable

PLM_DEVICE = None                  # None -> the adapter's own default

# ipSAE does not belong in this panel, and the default is None for that reason.
#
# It is not a scorer of variants: it is a co-folding CONFIDENCE measure, read
# off a predicted complex's PAE, and it exists in `track_a_chem_rank.py` to
# judge predictions - not to order point mutants of a solved structure. Running
# it here means folding 29 near-identical mutants of a crystallised Fab, and
# the panel then has almost no dynamic range for it: measured on those folds,
# the seed-to-seed spread per design (median 0.069) is LARGER than the spread
# between designs (sd 0.046). A test whose noise exceeds its signal cannot
# qualify or disqualify anything, so publishing an AUROC from it - 0.476, as it
# happens - invites the reading that this panel judged ipSAE and found it
# wanting. It did not. It cannot.
#
# The question that matters about ipSAE is whether it orders the 115 Track A
# designs reproducibly, and that is answered where the folds and the seeds
# actually are: the ICC in `06_score_correlation_audit.ipynb` (0.298 over 116
# designs x 3 seeds), which needs no negatives and no affinity labels. That
# number is what `qualification.csv` carries and what the ranker reads.
#
# Set this to a table from `examples/ipsae_panel.py` to add the column anyway;
# the cell below reports the two spreads beside the AUROC so the reading stays
# honest.
IPSAE_TABLE = None

# --- Part 2: the statistic --------------------------------------------------

BOOTSTRAP = 10_000                 # resamples for the CI
# The position-matched CI resamples within each position and so pays a pandas
# groupby per draw; 2,000 is enough for a percentile interval two decimals wide
# and keeps the cell under a minute.
MATCHED_BOOTSTRAP = 2_000
BOOTSTRAP_SEED = 20261103
CI = 0.95
MIN_POSITIVES = 3                  # below this, AUROC is NaN rather than a number
MIN_NEGATIVES = 2

# A permutation test is exact when the split is small enough to enumerate every
# assignment of the labels, and Monte Carlo above that. C(29,8) is 4.3 million,
# so the direction set is sampled; C(11,2) is 55, so the old test is exact.
EXACT_PERMUTATION_MAX = 200_000

# The old two-negative test, for comparison. Eleven ADAPT singles of which two
# were measured worse: the smallest p a one-sided exact rank test on that split
# can reach is 1/C(11,2), whatever the scorer does.
OLD_TEST_N = 11
OLD_TEST_NEGATIVES = 2

# --- Part 3: the Chinery binary ---------------------------------------------

CHINERY_STUDY = "chinery2026"
CHINERY_POSITIVE = "positive"              # fitted_KD, 156 wells
CHINERY_CLEAN = "negative_clean"           # bound_unfittable (I.C.), 97 wells
CHINERY_CAVEATED = "negative_caveated"     # source_NA, 434 wells
CHINERY_EXPECTED = {"positive": 156, "negative_clean": 97, "negative_caveated": 434}
CHINERY_SIZES = tuple(range(3, 11))        # the within-size AUROCs reported

# The caveated negatives are 434 more designs, 63% of Part 3's FoldX time, and
# their N/A label is ambiguous (non-binder, failed expression, or a sensor that
# did not regenerate - the source does not say which). Left False, Part 3 reports
# the clean split of 156 positives against 97 clean negatives and says the second
# table was not computed rather than guessing it. This said False in the comment
# and True in the value until 2026-11-21; the value is now what the comment says.
SCORE_CAVEATED = False

# --- Part 4: the manufactured negatives -------------------------------------

# The ten FoldX alanine-scan hotspots live in design_pipeline.hotspots, which
# holds the numbers quoted in docs/track_a_filtering.md. A hotspot is a position
# that loses at least HOTSPOT_DDG kcal/mol of binding on alanine.
HOTSPOT_DDG = 1.0

# The variable domains, which is where every manufactured variant is allowed to
# land. 1N8Z's chains are a whole light chain (214 residues) and a heavy Fd
# (220), so most of each chain is a constant domain that this pipeline never
# designs in and that no scorer here is calibrated on. The bounds are the ones
# design_pipeline.hotspots documents: VH 1 to 120, VL 1 to 107.
VARIABLE_DOMAIN = {HEAVY_CHAIN: 120, LIGHT_CHAIN: 107}

BURIED_BELOW = 0.15                # relative SASA under which a side chain is buried
SURFACE_ABOVE = 0.40               # relative SASA over which a side chain is exposed

# --- the three geometric classes of Part 4 ----------------------------------
# Selection criteria that owe nothing to FoldX, so FoldX can be graded on them.
# NEARMISS is the control on the controls: acidic positions far from the antigen
# and well exposed, given the same substitutions the measured improvements used.
# A scorer that calls those improvements is not reading the interface, it is
# reading the amino acid.
NEARMISS_DISTANCE = 10.0           # A, minimum heavy-atom distance to the antigen
NEARMISS_RSA = 0.40                # relative SASA above which the position is exposed
NEARMISS_ACIDS = ("D", "E")
NEARMISS_SUBS = ("F", "H", "K", "M", "W", "Q")   # the measured improvements' set

# MIRROR is the same chemistry pointed the wrong way: the residue classes that
# improved binding when installed at D31, D102 and D28, replaced by acids at the
# interface positions that already carry them.
MIRROR_DISTANCE = 5.0              # A, within which a position counts as interface
MIRROR_WT = ("K", "R", "F", "W", "Y")
MIRROR_SUBS = ("D", "E")

# GEOMETRIC hotspots: buried area on complex formation, which is a measurement
# of the coordinates rather than of an energy function. FoldX IS graded on these.
DSASA_ABOVE = 40.0                 # A^2 of side-chain area buried by the antigen
DSASA_TOP = 10                     # or the top n by buried area, whichever is larger

# The measured improvements at buried interface acids, which NEARMISS is scored
# against. Same substitutions, opposite structural context - so the comparison
# isolates the context. In 1N8Z's chain names.
NEARMISS_REFERENCE = (
    "B:D31R", "B:D31K", "B:D31M",
    "B:D102F", "B:D102H", "B:D102K", "B:D102M",
    "A:D28K", "A:D28R",
)
BULKY = "FWYLIMV"                  # the side chains glycine is a real loss for
SALT_BRIDGE_CUTOFF = 4.0           # angstrom, carboxyl oxygen to basic nitrogen
PARATOPE_CUTOFF = 5.0              # angstrom, the interface definition

DECOY_POSITIONS = 20               # non-interface surface positions sampled for decoys
DECOY_SEED = 20261103

# Each structure-derived class is capped and sampled with DECOY_SEED, because a
# Fv has more mid-strand positions and more salt bridges than a per-variant
# scorer can afford. The cap is a budget, not a judgement: raise it and the
# failure rates get tighter intervals at linear cost in FoldX hours.
MAX_PER_CLASS = 20

# What counts as a failure. `score_<model>` is higher-is-better-binding and zero
# is the wild type by construction for every scorer here, so a manufactured
# negative scored above zero is the scorer calling a deliberate break an
# improvement. The harder criterion is scoring above the median of the real
# positives from Part 1, which is what a shortlist cut would act on.
FAILURE_ABOVE = 0.0

MAX_VARIANTS = 1_000               # the cap score_variants applies to per-variant scorers
CACHE = True

In [2]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from design_pipeline import hotspots, metrics, numbering, preflight, structures
from design_pipeline.api import score_variants
from design_pipeline.errors import PipelineError
from design_pipeline.matrices import IndexMismatchError
from design_pipeline.structures import AMINO_ACIDS, Complex, chain_sequence
from design_pipeline.variants import enumerate_variants, make_id

diagnostics = metrics.diagnostics          # design_pipeline._diagnostics, loaded on use

pd.set_option("display.width", 185)
pd.set_option("display.max_columns", 50)

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

ALL_MODELS = tuple(MATRIX_MODELS) + tuple(MULTI_MODELS)
print(preflight.render(preflight.rows(ALL_MODELS)))
for note in preflight.notes(ALL_MODELS):
    print(f"\n{note}")
print(f"\nmatrix scorers, one substitution only: {', '.join(MATRIX_MODELS)}")
print(f"multi-capable scorers:                 {', '.join(MULTI_MODELS)}")
print("\nA scorer with no route here is reported as missing and costs you its row.")
print("Nothing below substitutes one scorer for another and nothing fills a")
print("column with zeros: an unscored cell stays empty and is counted as unscored.")

adapter       step   env        resolved               how
------------  -----  ---------  ---------------------  -----------------------------------------------------------------------------
antifold      score  by path    yes                    $DESIGN_PIPELINE_ANTIFOLD=/home/kira/protecher/tools/antifold/venv/bin/python
ablang2       score  core       yes                    import ablang2
esm1v         score  core       yes                    import transformers
esm2_650m     score  core       yes                    import transformers
antiberty     score  core       yes (route unchecked)  import antiberty
consensus     score  -          yes                    bundled
antifold_seq  score  by path    yes                    $DESIGN_PIPELINE_ANTIFOLD=/home/kira/protecher/tools/antifold/venv/bin/python
foldx         score  manual     yes                    $DESIGN_PIPELINE_FOLDX=/home/kira/protecher/.venv/bin/python
gearbind      score  container  NO                     not configured



## Part 1 - the labelled direction set

Two files, one label. `trastuzumab_mutation_evidence.csv` is one row per
*measurement*, so a variant owns several rows. `trastuzumab_negatives.csv` is
one row per *substitution*, so a triple owns three. Neither is a variant table,
and collapsing each to one row per variant is the first thing that happens here.

**Positives (21).** The 20 ADAPT 2017 Fab variants whose measured fold
improvement is above 1 - nine singles, seven doubles, four triples, built from
three sites: `H:D31` in CDR-H1, `H:D102` in CDR-H3 and `L:D28` in CDR-L1. The
ADAPT panel's fourth site, `L:S50`, appears only in its two losses, which is why
the positive half has three sites and 20 variants. Plus Moon 2016's `D98W`
(`H:D102W` in
mature-domain numbering), an IgG at 0.11 nM against a 0.48 nM wild type on the
same SPR run. Ten singles in all, then, which is what the matrix scorers get.

**Negatives (8).** The literature block of the negatives file, restricted to
measured affinity losses:

| variant | subs | format | what was measured |
|---|---|---|---|
| `L:S50F` | 1 | Fab | 0.3x the parent's affinity (ADAPT) |
| `L:S50W` | 1 | Fab | 0.5x the parent's affinity (ADAPT) |
| `L:R66G` | 1 | scFv | KD 4.8 +/- 0.7 nM against 2.0 +/- 0.4 nM |
| `L:V33L+L:R66G` | 2 | scFv | KD 4.9 nM against 2.0 nM |
| `L:V33I+L:R66G` | 2 | scFv | KD 5.2 nM against 2.0 nM |
| `AH12` | 3 | IgG | KD 1.12 nM against 0.48 nM |
| `AL07` | 3 | IgG | KD 0.79 nM against 0.48 nM |
| `A058` | 7 | IgG | KD 1.05 nM against 0.48 nM |

What gets dropped and why is printed below. The negatives file also carries
`negative_developability` - two triples heterogeneous on purification, one
double expressed too low to characterise - and `negative_function` - reduced
ADCC, reduced cell binding. Those are real failures and they are not a *binding*
direction. Grading an affinity scorer on a purification problem is grading it on
something it does not model.

**The deduplication.** `L:S50F` and `L:S50W` are in both files, and twice in the
negatives file: once in the literature block with evidence ids `TZM-062` and
`TZM-063`, once in the chinery block carried over from ADAPT with no id. The
positive selection excludes them by its own fold-change filter, so they can only
arrive as negatives, which is what they are. The assertion below is that exactly
these two collide and nothing else does - the two files are curated by hand, and
this is where a drift between them would surface.

In [3]:
# The benchmark's own notation, kept separate from `variant_id`: the first is
# (H:/L:, mature-domain 1-based), the second is this package's, over the chains
# 1N8Z actually has. The same three helpers as 03_adapt_calibration.ipynb and
# 03_chinery_size_calibration.ipynb, so the three notebooks parse one way.
TOKEN = re.compile(r"^(?P<chain>[A-Za-z0-9]+):(?P<wt>[A-Z])(?P<position>\d+)(?P<mut>[A-Z])$")


def parse_benchmark_variant(text):
    '''[(chain, position, wt, mut)] as the benchmark writes it.'''
    mutations = []
    for token in str(text).split(";"):
        match = TOKEN.match(token.strip())
        if match is None:
            raise ValueError(f"{text!r}: {token.strip()!r} is not a substitution like H:D31R")
        mutations.append((match["chain"], int(match["position"]), match["wt"], match["mut"]))
    if not mutations:
        raise ValueError(f"{text!r} names no substitution")
    return mutations


def remap(mutations):
    '''The same mutations over 1N8Z's chain ids. An unmapped chain is an error.'''
    out = []
    for chain, position, wt, mut in mutations:
        if chain not in CHAIN_MAP:
            raise KeyError(f"chain {chain!r} is not in CHAIN_MAP {CHAIN_MAP}")
        out.append((CHAIN_MAP[chain], position, wt, mut))
    return out


def label_of(mutations):
    '''The benchmark-side label: H:D31R+H:D102F, sorted, chains as written.'''
    return "+".join(f"{c}:{w}{p}{m}" for c, p, w, m in sorted(mutations))


evidence = pd.read_csv(EVIDENCE, low_memory=False)
negatives_file = pd.read_csv(NEGATIVES)
print(f"{EVIDENCE}\n  {len(evidence):,} rows, one per measurement")
print(f"{NEGATIVES}\n  {len(negatives_file):,} rows, one per substitution")

for frame, path, needed in (
    (evidence, EVIDENCE, ["evidence_id", "study", "variant", "format", "metric", "value",
                          "substitutions_mature_domain_1based", "outcome", "assay",
                          "parent_value", "uncertainty", "notes"]),
    (negatives_file, NEGATIVES, ["variant", "evidence_id", "label_class", "label_reason",
                                 "n_mutations", "chain", "position", "wt", "mut", "format",
                                 "study", "source_file"]),
):
    absent = [c for c in needed if c not in frame.columns]
    if absent:
        raise ValueError(f"{path} is missing columns this notebook reads: {absent}")

print(f"\n{Path(NEGATIVES).name} by block and label (distinct variants):")
print(negatives_file.groupby(["source_file", "label_class"])["variant"]
      .nunique().to_frame("variants").to_string())

data/benchmarks/trastuzumab_mutation_evidence.csv
  929 rows, one per measurement
data/benchmarks/trastuzumab_negatives.csv
  4,551 rows, one per substitution

trastuzumab_negatives.csv by block and label (distinct variants):
                                                   variants
source_file               label_class                      
chinery_block             negative_affinity               2
                          negative_caveated             434
                          negative_clean                 97
                          negative_developability         2
                          positive                      156
clearly_negative_variants negative_affinity               8
                          negative_developability         3
                          negative_function               2


In [4]:
# --- the positives ----------------------------------------------------------
# Selected by the measurement, not by a list of names, so a row that goes in
# comes with the number that put it there.
adapt = evidence[(evidence["study"] == ADAPT_STUDY) & (evidence["metric"] == ADAPT_METRIC)]
fold = pd.to_numeric(adapt["value"], errors="coerce")
if fold.isna().any():
    raise ValueError(f"{ADAPT_METRIC} rows with a non-numeric value: "
                     f"{sorted(adapt.loc[fold.isna(), 'variant'])}")
improved = adapt[fold > ADAPT_IMPROVED_ABOVE]
worsened = adapt[fold <= ADAPT_IMPROVED_ABOVE]

print(f"{ADAPT_STUDY} / {ADAPT_METRIC}: {len(adapt)} variants, "
      f"{len(improved)} above {ADAPT_IMPROVED_ABOVE}x, {len(worsened)} at or below")
print(f"  the {len(worsened)} at or below are {', '.join(sorted(worsened['variant']))}"
      f" - negatives, and they arrive from {Path(NEGATIVES).name}, not from here")

moon = evidence[evidence["evidence_id"] == MOON_POSITIVE]
if len(moon) != 1:
    raise ValueError(f"{MOON_POSITIVE} selects {len(moon)} rows in {EVIDENCE}, expected 1")
moon_row = moon.iloc[0]
print(f"\n{MOON_POSITIVE}: {moon_row['study']} {moon_row['variant']} "
      f"= {moon_row['substitutions_mature_domain_1based']}, {moon_row['format']}, "
      f"{moon_row['assay']}")
print(f"  KD {moon_row['value']} nM against a parent at {moon_row['parent_value']} nM "
      f"-> {moon_row['outcome']}")

records = []
for row in improved.itertuples():
    mutations = parse_benchmark_variant(row.substitutions_mature_domain_1based)
    records.append({
        "evidence_id": row.evidence_id, "source": ADAPT_STUDY,
        "label": label_of(mutations), "format": row.format,
        "direction": 1, "n_mutations": len(mutations),
        "evidence": f"{row.value}x fold improvement", "assay": row.assay,
        "mutations": tuple(sorted(remap(mutations))),
    })
moon_mutations = parse_benchmark_variant(moon_row["substitutions_mature_domain_1based"])
records.append({
    "evidence_id": moon_row["evidence_id"], "source": moon_row["study"],
    "label": label_of(moon_mutations), "format": moon_row["format"],
    "direction": 1, "n_mutations": len(moon_mutations),
    "evidence": f"KD {moon_row['value']} vs parent {moon_row['parent_value']} nM",
    "assay": moon_row["assay"],
    "mutations": tuple(sorted(remap(moon_mutations))),
})
positives = pd.DataFrame(records)

if len(positives) != EXPECTED_POSITIVES:
    raise ValueError(f"expected {EXPECTED_POSITIVES} positives, built {len(positives)}")
sizes = positives["n_mutations"].value_counts().to_dict()
if sizes != EXPECTED_POSITIVE_SIZES:
    raise ValueError(f"expected {EXPECTED_POSITIVE_SIZES} positives by size, got {sizes}")
print(f"\n{len(positives)} positives: "
      + ", ".join(f"{n} with {k} substitution(s)" for k, n in sorted(sizes.items())))

adapt2017 / affinity_fold_improvement: 22 variants, 20 above 1.0x, 2 at or below
  the 2 at or below are L:S50F, L:S50W - negatives, and they arrive from trastuzumab_negatives.csv, not from here

TZM-019: moon2016 D98W = H:D102W, IgG, SPR
  KD 0.11 nM against a parent at 0.48 nM -> improved affinity

21 positives: 10 with 1 substitution(s), 7 with 2 substitution(s), 4 with 3 substitution(s)


In [5]:
# --- the negatives ----------------------------------------------------------
# Selected by label and not by block, because `negative_affinity` is in both
# blocks: the literature block has the eight with evidence ids, and the chinery
# block carries two more over from ADAPT with no id at all. Both are taken here
# so that the deduplication in the next cell has something to do - filtering to
# one block would hide the collision instead of printing it.
#
# One row per substitution, so a variant is a group, and the group key is
# (variant, study) rather than evidence_id for the same reason: two of the
# groups have no id.
affinity = negatives_file[negatives_file["label_class"] == NEGATIVE_LABEL]
literature = negatives_file[negatives_file["source_file"] == NEGATIVE_SOURCE]
other = literature[literature["label_class"] != NEGATIVE_LABEL]

print(f"{NEGATIVE_LABEL} across the whole file: {affinity['variant'].nunique()} "
      f"distinct variant names in {len(affinity)} substitution rows, by block:")
print(affinity.groupby(["source_file", "study"])["variant"].nunique()
      .to_frame("variants").to_string())
print(f"\n{NEGATIVE_SOURCE}: {literature['variant'].nunique()} variants, of which")
print(f"  {literature[literature['label_class'] == NEGATIVE_LABEL]['variant'].nunique()}"
      f" are {NEGATIVE_LABEL} and go in")
print(f"  {other['variant'].nunique()} are not a binding direction and are dropped:")
for (variant, label, reason), _ in other.groupby(
    ["variant", "label_class", "label_reason"], sort=True
):
    print(f"    {variant:<30}{label:<26}{reason}")
print("  A purification or ADCC failure is a real failure. It is not a measured")
print("  change in affinity, and an affinity scorer graded on it is graded on")
print("  something it does not model.")

records = []
for (variant, study), group in affinity.groupby(["variant", "study"], sort=True):
    mutations = [
        (str(r.chain), int(r.position), str(r.wt), str(r.mut)) for r in group.itertuples()
    ]
    declared = sorted(set(int(n) for n in group["n_mutations"]))
    if declared != [len(mutations)]:
        raise ValueError(
            f"{variant}: {len(mutations)} substitution rows but n_mutations says "
            f"{declared}. The file's own count disagrees with its own rows, so one "
            f"of the two is wrong and the sizes below cannot be trusted."
        )
    formats = sorted(set(group["format"]))
    if len(formats) != 1:
        raise ValueError(f"{variant}: more than one format on one variant: {formats}")
    ids = sorted({i for i in group["evidence_id"] if isinstance(i, str) and i})
    records.append({
        "evidence_id": ids[0] if ids else "",
        "source": study, "well": variant, "label": label_of(mutations),
        "format": formats[0], "direction": 0, "n_mutations": len(mutations),
        "evidence": sorted(set(group["label_reason"]))[0], "assay": "",
        "mutations": tuple(sorted(remap(mutations))),
    })
raw_negatives = pd.DataFrame(records)
print(f"\n{len(raw_negatives)} {NEGATIVE_LABEL} variants before deduplication")
print(raw_negatives[["well", "label", "n_mutations", "format", "source",
                     "evidence_id", "evidence"]].to_string(index=False))

negative_affinity across the whole file: 8 distinct variant names in 22 substitution rows, by block:
                                      variants
source_file               study               
chinery_block             adapt2017          2
clearly_negative_variants literature         8

clearly_negative_variants: 13 variants, of which
  8 are negative_affinity and go in
  5 are not a binding direction and are dropped:
    H:D102F;H:D31R;L:D28K         negative_developability   heterogeneous on purification
    H:D102F;H:D31R;L:D28R         negative_developability   heterogeneous on purification
    VH-A40R/K43D                  negative_developability   too low for characterization
    VH-R50S/R59N                  negative_function         cell_binding: significantly reduced
    VL-Q89A                       negative_function         ADCC: reduced
  A purification or ADCC failure is a real failure. It is not a measured
  change in affinity, and an affinity scorer graded on it is gra

In [6]:
# --- deduplicate, on the mutation set and nothing else ----------------------
# `label` is the key because it is derived from the substitutions, so two rows
# naming the same variant in different notation still collide. The assertion is
# that the collisions are exactly EXPECTED_DUPLICATES.
collisions = (
    raw_negatives.groupby("label").filter(lambda g: len(g) > 1).sort_values(["label", "source"])
)
found = tuple(sorted(collisions["label"].unique()))
print(f"labels appearing more than once: {', '.join(found) if found else '(none)'}")
if not collisions.empty:
    print(collisions[["label", "source", "well", "evidence_id", "evidence", "format"]]
          .to_string(index=False))
if found != tuple(sorted(EXPECTED_DUPLICATES)):
    raise ValueError(
        f"expected the duplicates to be exactly {sorted(EXPECTED_DUPLICATES)}, found "
        f"{list(found)}. Both files are curated by hand and this is where a drift "
        f"between them shows up; reconcile the two before reading any AUROC below."
    )

# Keep the copy carrying an evidence id, so the surviving row points back at a
# measurement rather than at a label carried over from another study.
raw_negatives = raw_negatives.assign(has_id=raw_negatives["evidence_id"].astype(bool))
negatives = (
    raw_negatives.sort_values(["label", "has_id"], ascending=[True, False])
    .drop_duplicates("label", keep="first")
    .drop(columns=["has_id", "well"])
    .reset_index(drop=True)
)
if len(negatives) != EXPECTED_NEGATIVES:
    raise ValueError(f"expected {EXPECTED_NEGATIVES} negatives, kept {len(negatives)}")
sizes = negatives["n_mutations"].value_counts().to_dict()
if sizes != EXPECTED_NEGATIVE_SIZES:
    raise ValueError(f"expected {EXPECTED_NEGATIVE_SIZES} negatives by size, got {sizes}")
print(f"\n{len(negatives)} negatives after deduplication: "
      + ", ".join(f"{n} with {k} substitution(s)" for k, n in sorted(sizes.items())))

# The cross-check deduplication cannot make: a label must not be in both halves.
# If one paper measured a variant better and another measured it worse, that is a
# finding, not two rows to average.
overlap = sorted(set(positives["label"]) & set(negatives["label"]))
if overlap:
    raise ValueError(
        f"{len(overlap)} label(s) are both positive and negative: {overlap}. Two "
        f"measurements disagree about the direction and this notebook cannot decide "
        f"which to believe."
    )
print(f"no label is in both halves ({positives['label'].nunique()} positive labels, "
      f"{negatives['label'].nunique()} negative)")

direction = (
    pd.concat([positives, negatives], ignore_index=True)
    .assign(variant_id=lambda f: [make_id(m) for m in f["mutations"]])
    .assign(multi=lambda f: f["n_mutations"] > 1)
    .sort_values(["direction", "n_mutations", "label"], ascending=[False, True, True])
    .reset_index(drop=True)
)
if direction["variant_id"].duplicated().any():
    raise ValueError("a variant_id appears twice after the concat")

labels appearing more than once: L:S50F, L:S50W
 label     source   well evidence_id                   evidence format
L:S50F  adapt2017 L:S50F             measured_worse_than_parent    Fab
L:S50F literature L:S50F     TZM-062                   fold 0.3    Fab
L:S50W  adapt2017 L:S50W             measured_worse_than_parent    Fab
L:S50W literature L:S50W     TZM-063                   fold 0.5    Fab

8 negatives after deduplication: 3 with 1 substitution(s), 2 with 2 substitution(s), 2 with 3 substitution(s), 1 with 7 substitution(s)
no label is in both halves (21 positive labels, 8 negative)


In [7]:
# --- the counts per cell, which is the point of printing anything at all ----
print(f"the direction set: {len(direction)} variants, "
      f"{int(direction['direction'].sum())} improved, "
      f"{int((1 - direction['direction']).sum())} worsened\n")


def per_cell(column, name):
    '''improved / worsened / total, one row per value of `column`.'''
    table = pd.crosstab(
        direction[column], direction["direction"].map({1: "improved", 0: "worsened"})
    ).reindex(columns=["improved", "worsened"], fill_value=0)
    table.index.name = name
    table["total"] = table.sum(axis=1)
    table.loc["ALL"] = table.sum()
    return table


for column, name in (("n_mutations", "substitutions"), ("format", "format"),
                     ("source", "study"), ("multi", "several subs")):
    print(per_cell(column, name).to_string(), "\n")

print("Read the format table twice. The scFv rows are negatives only and the IgG")
print("rows are one positive against three negatives, so a by-format AUROC exists")
print("for Fab and for nothing else - which Part 2 reports rather than papers over.")
print()
print(direction[["label", "direction", "n_mutations", "format", "source", "evidence"]]
      .assign(direction=lambda f: f["direction"].map({1: "improved", 0: "worsened"}))
      .to_string(index=False))

# The positions, because they are reused and reuse is non-independence.
sites = {}
for row in direction.itertuples():
    for chain, position, wt, _mut in row.mutations:
        key = (chain, position, wt)
        entry = sites.setdefault(key, {"improved": 0, "worsened": 0})
        entry["improved" if row.direction == 1 else "worsened"] += 1
print(f"\n{len(sites)} distinct sites over {len(direction)} variants:")
print(f"{'site (1N8Z)':<14}{'in improved':>13}{'in worsened':>13}")
print("-" * 40)
for (chain, position, wt), counts in sorted(sites.items()):
    print(f"{f'{chain}:{wt}{position}':<14}{counts['improved']:>13}{counts['worsened']:>13}")
print("-" * 40)
positive_sites = {k for k, v in sites.items() if v["improved"]}
negative_sites = {k for k, v in sites.items() if v["worsened"]}
print(f"{len(positive_sites)} site(s) carry all {int(direction['direction'].sum())} "
      f"improvers and {len(negative_sites)} carry all "
      f"{int((1 - direction['direction']).sum())} losses; "
      f"{len(positive_sites & negative_sites)} site(s) are in both.")
print(f"{direction['source'].nunique()} studies in all. The bootstrap in Part 2")
print("resamples variants, not sites and not studies, so it does not know about")
print("any of this and the interval it prints is therefore the optimistic one.")

path = out_dir / "direction_set.csv"
direction.drop(columns=["mutations"]).to_csv(path, index=False)
print(f"\n-> {path}")

the direction set: 29 variants, 21 improved, 8 worsened

direction      improved  worsened  total
substitutions                           
1                    10         3     13
2                     7         2      9
3                     4         2      6
7                     0         1      1
ALL                  21         8     29 

direction  improved  worsened  total
format                              
Fab              20         2     22
IgG               1         3      4
scFv              0         3      3
ALL              21         8     29 

direction   improved  worsened  total
study                                
adapt2017         20         0     20
literature         0         8      8
moon2016           1         0      1
ALL               21         8     29 

direction     improved  worsened  total
several subs                           
False               10         3     13
True                11         5     16
ALL                 21         8     29 

### Direction only, never KD, and the table above is the reason

Three formats and three assays. ADAPT is **Fab**, monovalent, SPR, reported as a
fold improvement against a batch-matched parent run alongside. Moon is **IgG**,
two Fab arms, SPR, an absolute KD against a wild-type well on the same run.
Okazaki is **scFv**, a different construct again, with its own parent and a
melting temperature that the paper reports moving.

**The sign is comparable across all three. The magnitude is not.** A 2.8-fold
Fab improvement and a 4.4-fold IgG improvement are not 2.8 and 4.4 on one scale:
the IgG number carries an avidity term the Fab number does not have, the scFv
number carries a construct whose stability changed, and the SPR runs were
normalised against different parents. So **nothing in this notebook pools a KD,
a fold change or a ddG across those rows, and nothing correlates against one.**
The only thing asked of a label is which side of its own parent the variant
landed on, which is the one property all three assays agree on how to report.

That is also why the statistic is AUROC and not Spearman. Spearman over the
pooled set would be ranking a Fab fold change against an IgG KD ratio and
calling the result a calibration. AUROC only ever compares pairs with opposite
labels, so it never puts two magnitudes on one axis at all.

**Three things about these eight negatives, out loud.**

* **`L:R66G` is a weak negative.** The source describes the affinity as
  *similar* and reports **4.8 +/- 0.7 nM against 2.0 +/- 0.4 nM with no
  significance test**; the curator kept the numerical increase and said so in
  the row's `notes`. It is in the set because dropping it would leave two single
  negatives, which is the hole this notebook exists to fill. It is flagged here
  because a 2.4-fold shift with overlapping stated uncertainties and no test is
  not the evidence that ADAPT's 0.3x is. Part 2 reports every AUROC again with
  it dropped, so the weight it carries is visible rather than argued about.
* **Three of the eight share one substitution.** `L:R66G` is the whole of one
  negative and half of the other two, so the negative half is nearer five
  independent observations than eight. The bootstrap resamples variants, so it
  does not know this; the interval it prints is the optimistic one.
* **`A058` contains `H:D102W`, which is also the whole of a positive.** Moon's
  `D98W` is `H:D102W` and improved affinity 4.4-fold. `A058` is `H:D102W` plus
  six more substitutions and worsened it. Moon's own `AH06` -
  `H:D102W+H:M107F+H:D108A+H:Y109L`, 8-fold improved, deliberately not in this
  set because the set is the one specified - differs from `A058` only by three VL
  substitutions. So one substitution is the best single in the panel, a
  component of an 8-fold improvement, and a component of a loss, depending on
  what sits beside it. **No scorer that treats substitutions as independent can
  get all three right**, and that is a ceiling on the matrix scorers that no
  amount of extra data moves.

## Part 2 - direction AUROC per scorer

The numbering gate first, then one scoring call per scorer, then the statistic.

**The gate.** Both files number their substitutions into the mature domain,
1-based; a `variant_id` here is 1-based into *the chain as given in the
structure*. Those agree only if 1N8Z's chains start at domain residue 1 with no
gaps, which is not a thing to assume: a set numbered in Kabat or IMGT would land
every substitution a few residues off and still score perfectly cleanly, and the
AUROC would be a real number about the wrong residues. The check is the simple
one - the residue each file says is wild type has to be the residue 1N8Z has at
that chain and position - run over every site in the set, and then made a second
time by `enumerate_variants`, which re-checks on the way in. Nothing after it
runs if it fails.

**The statistic.** AUROC computed from ranks, so ties count half, which matters:
a matrix scorer gives the same number to every candidate at a position it has no
opinion about. Three readings of it, in order:

* **Overall**, per scorer, over whatever that scorer could answer.
* **Singles against multi-mutation.** The matrix scorers see 10 positives and 3
  negatives and nothing else; the multi-capable scorers see all 21 and all 8.
  These are not the same test and the numbers are not comparable across the
  split, which is why they are two columns and not one.
* **By format.** If a scorer's errors cluster in one format, the pooled number
  is reporting an assay artifact. Part 1's format table says in advance that
  only Fab has both labels, so this reading is mostly a check that the IgG and
  scFv rows are not quietly carrying the result.

**The interval.** A stratified bootstrap - positives resampled from positives,
negatives from negatives, `BOOTSTRAP` times - and the percentile interval. With
21 against 8 that interval is about 0.4 wide, and it is printed next to every
point estimate for exactly that reason. It is also optimistic: it resamples
variants, and Part 1 showed the variants share fifteen sites and three studies -
with all 21 improvers built from three of those sites.

**And the old test, for comparison.** Before these negatives existed the whole
of the direction evidence was 11 ADAPT singles of which 2 were measured worse.
The smallest p a one-sided exact rank test on that split can reach is
**1/C(11,2) = 1/55 = 0.018**, and that is the *best possible* outcome - a scorer
that ranked both losses dead last could not do better. It is reported below
beside the real p values so that what the extra six negatives bought is a
number: the floor moves from 0.018 to 1/C(29,8), which is 2.3e-7.

In [8]:
CANDIDATES = (
    Path("data/structures/1N8Z.pdb"),
    Path("data/1N8Z.pdb"),
    Path.home() / ".cache" / "design_pipeline" / "pdb" / "1n8z.pdb",
)


def resolve_pdb():
    if PDB_PATH is not None:
        path = Path(PDB_PATH).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f"PDB_PATH does not exist: {path}")
        return path
    for candidate in CANDIDATES:
        if candidate.is_file():
            return candidate
    looked = "\n".join(f"  {c}" for c in CANDIDATES)
    raise FileNotFoundError(
        f"no local 1N8Z found. Looked in:\n{looked}\nSet PDB_PATH to the copy you "
        f"mean to use. Nothing here downloads: a structure that arrived by another "
        f"route is a different structure, and every position below is an index into "
        f"this one file."
    )


pdb = resolve_pdb()
chains_present = structures.chain_ids(pdb)
roles = {"heavy": HEAVY_CHAIN, "light": LIGHT_CHAIN, "antigen": ANTIGEN_CHAIN}
unknown = {role: chain for role, chain in roles.items() if chain not in chains_present}
if unknown:
    raise KeyError(f"chains named in the parameters that {pdb} does not have: "
                   f"{unknown}. Present: {chains_present}")
sequences = {chain: chain_sequence(pdb, chain) for chain in chains_present}
residue_ids = {chain: structures.chain_residue_ids(pdb, chain) for chain in chains_present}

print(f"structure: {pdb}")
print(f"{'chain':<7}{'role':<9}{'len':>5}{'first':>8}{'last':>8}{'labels 1..n':>14}")
print("-" * 52)
role_of = {v: k for k, v in roles.items()}
contiguous = {}
for chain in chains_present:
    labels = residue_ids[chain]
    numeric = all(label.isdigit() for label in labels)
    runs = numeric and [int(l) for l in labels] == list(range(1, len(labels) + 1))
    contiguous[chain] = runs
    print(f"{chain:<7}{role_of.get(chain, '-'):<9}{len(sequences[chain]):>5}"
          f"{labels[0]:>8}{labels[-1]:>8}{('yes' if runs else 'NO'):>14}")

# --- the gate: every site in the direction set, against the coordinates -----
sites = {}
for row in direction.itertuples():
    for chain, position, wt, _mut in row.mutations:
        sites.setdefault((chain, position, wt), []).append(row.label)

print(f"\n{'site (1N8Z)':<14}{'files say wt':>14}{'1N8Z has':>10}{'pdb label':>11}"
      f"{'variants':>10}  verdict")
print("-" * 73)
mismatches = []
for (chain, position, wt), used_by in sorted(sites.items()):
    sequence = sequences[chain]
    inside = 0 < position <= len(sequence)
    here = sequence[position - 1] if inside else "-"
    label = residue_ids[chain][position - 1] if inside else "-"
    ok = inside and here == wt
    if not ok:
        mismatches.append((chain, position, wt, here))
    print(f"{f'{chain}:{position}':<14}{wt:>14}{here:>10}{label:>11}{len(used_by):>10}  "
          f"{'ok' if ok else 'MISMATCH'}")
if mismatches:
    raise IndexMismatchError(
        "the numbering does not line up with the structure: "
        + "; ".join(f"{c}:{p} the files say {w}, 1N8Z has {h}" for c, p, w, h in mismatches)
        + ". Every AUROC below would be a real number for the wrong residues. Check "
        "CHAIN_MAP and each block's numbering before going on."
    )
NUMBERING_OK = True
print(f"\n{len(sites)} sites, {len(direction)} variants, every wild-type residue agrees")
print(f"(1N8Z's own labels run 1..n on chains {HEAVY_CHAIN} and {LIGHT_CHAIN}: "
      f"{contiguous[HEAVY_CHAIN] and contiguous[LIGHT_CHAIN]}, so the files' "
      f"mature-domain positions and this package's are the same positions here)")

structure: data/structures/1N8Z.pdb
chain  role       len   first    last   labels 1..n
----------------------------------------------------
A      light      214       1     214           yes
B      heavy      220       1     220           yes
C      antigen    581       1     607            NO

site (1N8Z)     files say wt  1N8Z has  pdb label  variants  verdict
-------------------------------------------------------------------------
A:28                       D         D         28         6  ok
A:33                       V         V         33         2  ok
A:50                       S         S         50         2  ok
A:66                       R         R         66         3  ok
A:93                       T         T         93         2  ok
A:95                       P         P         95         1  ok
A:96                       P         P         96         2  ok
A:97                       T         T         97         1  ok
B:31                       D         D         

In [9]:
# --- one scoring helper, used by Parts 2, 3 and 4 ---------------------------
# `score_variants` sends **kwargs to every model in a call, so these three want
# separate calls: AntiFold needs chain ids and a source folder, FoldX reads its
# binary from the environment, the language models take a device. One call per
# model, each joining its column into the table the previous one returned.
complex_ = Complex(
    paths=(pdb,),
    output_dir=out_dir,
    config={"model_name": "local_file", "source": str(pdb)},
)

for name, setting in (("FOLDX_RUNS", FOLDX_RUNS), ("FOLDX_WORKERS", FOLDX_WORKERS)):
    if setting is not None:
        os.environ[name] = str(setting)

if GEARBIND_IMAGE is not None:
    from dataclasses import replace

    from design_pipeline import REGISTRY
    from design_pipeline.adapters import _scorers

    REGISTRY["gearbind"] = replace(
        _scorers.gearbind, tool=_scorers.matrix_container(GEARBIND_IMAGE)
    )
    print(f"gearbind -> {GEARBIND_IMAGE}")

MODEL_KWARGS = {
    "antifold": {"executable": ANTIFOLD_PYTHON, "heavy": HEAVY_CHAIN,
                 "light": LIGHT_CHAIN, "antifold_src": ANTIFOLD_SRC},
    "antifold_seq": {"executable": ANTIFOLD_PYTHON, "heavy": HEAVY_CHAIN,
                     "light": LIGHT_CHAIN, "device": ANTIFOLD_DEVICE,
                     "weights": ANTIFOLD_WEIGHTS},
    "foldx": {"executable": FOLDX_PYTHON},
    "ablang2": {"device": PLM_DEVICE, "antibody_chains": [HEAVY_CHAIN, LIGHT_CHAIN]},
    "esm1v": {"device": PLM_DEVICE, "antibody_chains": [HEAVY_CHAIN, LIGHT_CHAIN]},
    "esm2_650m": {"device": PLM_DEVICE, "antibody_chains": [HEAVY_CHAIN, LIGHT_CHAIN]},
    "antiberty": {"device": PLM_DEVICE, "antibody_chains": [HEAVY_CHAIN, LIGHT_CHAIN]},
}


def score_set(table, models, where, label):
    '''(table, ran, skipped) with one `score_<model>` column per scorer that ran.

    A scorer that cannot run here is recorded with the reason the package gave
    and the rest carry on: a missing FoldX licence should cost you the FoldX row
    of the verdict table, not the other eight. A scorer that runs and answers
    nothing is also a skip, because an all-empty column is not a result.
    '''
    ran, skipped = [], {}
    for model in models:
        kwargs = {k: v for k, v in MODEL_KWARGS.get(model, {}).items() if v is not None}
        try:
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", message="anarci is not installed")
                table = score_variants(
                    complex_, table, where, models=[model], structure=pdb,
                    max_variants=MAX_VARIANTS, cache=CACHE, **kwargs,
                )
        except PipelineError as exc:
            skipped[model] = f"{type(exc).__name__}: {exc}"
            print(f"  {model:<14} SKIPPED  {type(exc).__name__}: "
                  f"{str(exc).splitlines()[0]}")
            continue
        column = f"score_{model}"
        one = table.frame.drop_duplicates("variant_id")
        answered = int(one[column].notna().sum()) if column in one.columns else 0
        if answered == 0:
            skipped[model] = "ran, but scored nothing"
            print(f"  {model:<14} ran and scored nothing, so treated as skipped")
            continue
        ran.append(model)
        print(f"  {model:<14} {answered:>5} of {table.candidates} candidates scored")
    print(f"{label}: scorers that ran: {', '.join(ran) if ran else 'none'}")
    return table, ran, skipped


if not NUMBERING_OK:
    raise RuntimeError("Part 2's numbering gate did not pass")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="anarci is not installed")
    direction_table = enumerate_variants(
        complex_, HEAVY_CHAIN, out_dir / "direction" / "variants",
        only=direction["variant_id"].tolist(), structure=pdb,
    )
print(f"{direction_table.rows} rows, {direction_table.candidates} candidates "
      f"-> {direction_table.path}")
if direction_table.candidates != len(direction):
    raise ValueError(f"{direction_table.candidates} candidates for {len(direction)} variants")

print("\nscoring the direction set:")
direction_scored, direction_ran, direction_skipped = score_set(
    direction_table, ALL_MODELS, out_dir / "direction" / "scores", "direction set"
)
if not direction_ran:
    raise RuntimeError(
        "no scorer ran, so there is no AUROC to report. Part 0's table says what "
        "each one needs; the cheapest to satisfy are the language models, which "
        "install into this environment with the `plm` extra."
    )

56 rows, 29 candidates -> runs/negative_controls/direction/variants/requested.parquet

scoring the direction set:
  antifold          13 of 29 candidates scored
  ablang2           13 of 29 candidates scored
  esm1v             13 of 29 candidates scored
  esm2_650m         13 of 29 candidates scored
  antiberty         13 of 29 candidates scored
  consensus      SKIPPED  MissingDependencyError: the consensus scorer needs an OAS frequency table and none is configured. Pass frequencies=<path>, or set DESIGN_PIPELINE_OAS_TABLE. The file is a CSV of imgt,aa,frequency (or position,aa,frequency), built once from the Observed Antibody Space download at https://opig.stats.ox.ac.uk/webapps/oas/. Drop 'consensus' from models=[...] to run without it.
  antifold_seq      29 of 29 candidates scored


  foldx             29 of 29 candidates scored
  gearbind       SKIPPED  MissingRuntimeError: gearbind is not installed. Verified route (checked 2026-10-30) [container]: build an image from https://github.com/DeepGraphLearning/GearBind and set it with REGISTRY['gearbind'] - no published image exists. Note: TorchDrug pins Python 3.7-3.10 and an old torch: the one genuine conflict in this set, hence a container rather than Env B.. `python -m design_pipeline.preflight` prints the whole table.
direction set: scorers that ran: antifold, ablang2, esm1v, esm2_650m, antiberty, antifold_seq, foldx


In [10]:
# --- AUROC, its bootstrap interval, and a permutation p ---------------------
# AUROC from averaged ranks, which is what makes ties count half. That is not a
# detail here: a matrix scorer returns the same number at every position it has
# no opinion about, and counting those pairs as wins would be a free half of the
# statistic.
RNG = np.random.default_rng(BOOTSTRAP_SEED)


def auroc(scores, labels):
    '''(auroc, n_pos, n_neg) over the finite scores. NaN when the cells are too thin.

    NaN rather than 0.5, because 0.5 is a claim - "this scorer is a coin" - and
    the absence of enough data to make a claim is not that claim.
    '''
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    usable = np.isfinite(scores)
    s, y = scores[usable], labels[usable]
    n_pos, n_neg = int((y == 1).sum()), int((y == 0).sum())
    if n_pos < MIN_POSITIVES or n_neg < MIN_NEGATIVES:
        return float("nan"), n_pos, n_neg
    ranks = metrics.rankdata(s)
    u = float(ranks[y == 1].sum()) - n_pos * (n_pos + 1) / 2.0
    return u / float(n_pos * n_neg), n_pos, n_neg


def auroc_ci(scores, labels, resamples=None):
    '''Percentile CI from a stratified bootstrap: positives and negatives separately.

    Stratified because the class sizes are the design of the panel, not a draw
    from anything - resampling the pooled set would sometimes produce a draw
    with no negatives at all, and the interval would then be an interval over a
    different experiment.
    '''
    resamples = BOOTSTRAP if resamples is None else resamples
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    usable = np.isfinite(scores)
    s, y = scores[usable], labels[usable]
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    if len(pos) < MIN_POSITIVES or len(neg) < MIN_NEGATIVES:
        return float("nan"), float("nan")
    draws = np.empty(resamples, dtype=float)
    for i in range(resamples):
        take = np.concatenate([
            RNG.choice(pos, size=len(pos), replace=True),
            RNG.choice(neg, size=len(neg), replace=True),
        ])
        draws[i] = auroc(s[take], y[take])[0]
    draws = draws[np.isfinite(draws)]
    if not len(draws):
        return float("nan"), float("nan")
    low, high = (1.0 - CI) / 2.0, 1.0 - (1.0 - CI) / 2.0
    return float(np.quantile(draws, low)), float(np.quantile(draws, high))


def permutation_p(scores, labels, resamples=None):
    '''One-sided P(AUROC >= observed) under the null, and whether it was exact.

    Exact by enumeration when the number of label assignments is small enough,
    Monte Carlo above that, with the usual (hits + 1) / (draws + 1) so the
    reported p is never zero. `floor` is the smallest p this design can reach -
    1/C(n, n_neg) exact, 1/(resamples + 1) sampled - which is the number that
    makes a small panel's significance legible.
    '''
    from itertools import combinations
    from math import comb

    resamples = BOOTSTRAP if resamples is None else resamples
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    usable = np.isfinite(scores)
    s, y = scores[usable], labels[usable]
    n, n_neg = len(y), int((y == 0).sum())
    observed = auroc(s, y)[0]
    if observed != observed:
        return float("nan"), float("nan"), "too few"
    total = comb(n, n_neg)
    if total <= EXACT_PERMUTATION_MAX:
        hits = 0
        for picked in combinations(range(n), n_neg):
            shuffled = np.ones(n, dtype=int)
            shuffled[list(picked)] = 0
            if auroc(s, shuffled)[0] >= observed - 1e-12:
                hits += 1
        return hits / float(total), 1.0 / float(total), f"exact over {total:,}"
    hits = 0
    for _ in range(resamples):
        if auroc(s, RNG.permutation(y))[0] >= observed - 1e-12:
            hits += 1
    return (hits + 1) / float(resamples + 1), 1.0 / float(resamples + 1), \
        f"sampled, {resamples:,} draws"


def cell(value, width=0):
    text = "-" if value != value else f"{value:.3f}"
    return f"{text:>{width}}" if width else text


# The old test's ceiling, computed rather than quoted.
from math import comb

old_floor = 1.0 / comb(OLD_TEST_N, OLD_TEST_NEGATIVES)
new_floor = 1.0 / comb(len(direction), int((1 - direction["direction"]).sum()))
print(f"the old two-negative test: {OLD_TEST_N} ADAPT singles, "
      f"{OLD_TEST_NEGATIVES} of them measured worse")
print(f"  best possible p = 1/C({OLD_TEST_N},{OLD_TEST_NEGATIVES}) = "
      f"1/{comb(OLD_TEST_N, OLD_TEST_NEGATIVES)} = {old_floor:.4f}")
print(f"  a scorer that put both losses dead last could not beat that, so nothing")
print(f"  measured on it could ever have reached p < 0.01.")
print(f"\nthis set: {len(direction)} variants, "
      f"{int((1 - direction['direction']).sum())} of them measured worse")
print(f"  best possible p = 1/C({len(direction)},"
      f"{int((1 - direction['direction']).sum())}) = {new_floor:.2e}")
print(f"  the floor drops by a factor of {old_floor / new_floor:,.0f}. That is what")
print(f"  the extra six negatives bought, and it is a bound on what is provable,")
print(f"  not evidence that anything is true.")


# An alias, because `cell` is an easy name to shadow in a notebook.
cell_text = cell


the old two-negative test: 11 ADAPT singles, 2 of them measured worse
  best possible p = 1/C(11,2) = 1/55 = 0.0182
  a scorer that put both losses dead last could not beat that, so nothing
  measured on it could ever have reached p < 0.01.

this set: 29 variants, 8 of them measured worse
  best possible p = 1/C(29,8) = 2.33e-07
  the floor drops by a factor of 78,039. That is what
  the extra six negatives bought, and it is a bound on what is provable,
  not evidence that anything is true.


In [11]:
# --- ipSAE as another scorer column, if its folds exist ---------------------
# The point of merging rather than computing an AUROC here: the interval, the
# permutation p and the tie handling then come from the same functions as the
# other seven, so the ipSAE row is comparable to them by construction instead
# of by inspection. The score is the 3-seed MEAN of the larger chain direction
# (`ab` aligns on the scFv and scores the antigen); the max over seeds is the
# largest of k draws and biased upward by about 0.85 SD at k = 3.
IPSAE_RAN = False
if IPSAE_TABLE:
    ipsae_path = Path(IPSAE_TABLE)
    if not ipsae_path.is_file():
        print(f"no ipSAE table at {ipsae_path}, so ipSAE is not in this panel.")
        print("examples/ipsae_panel.py fold --panel direction writes it; it needs")
        print("the parent alignments and a GPU.")
    else:
        ipsae_table = pd.read_csv(ipsae_path)
        missing = sorted(set(direction["variant_id"]) - set(ipsae_table["variant_id"]))
        # A Series, joined by the cell below rather than written into
        # `direction_scored.frame`: `VariantTable.frame` is a property that
        # re-reads the file on every access and caches nothing, so a column
        # assigned to it lands on a throwaway copy and the next read quietly
        # does not have it.
        IPSAE_SCORES = ipsae_table.set_index("variant_id")["score_ipsae"]
        MULTI_MODELS = tuple(MULTI_MODELS) + ("ipsae",)
        ALL_MODELS = tuple(MATRIX_MODELS) + tuple(MULTI_MODELS)
        IPSAE_RAN = True
        seeds = sorted(set(ipsae_table["seeds"]))
        print(f"ipSAE: {len(ipsae_table)} designs from {ipsae_path}, "
              f"{seeds} seed(s) each")
        if missing:
            print(f"{len(missing)} design(s) of the panel have no fold: "
                  f"{', '.join(missing[:5])}{' ...' if len(missing) > 5 else ''}")
        spread = pd.to_numeric(ipsae_table.get("seed_spread"), errors="coerce").dropna()
        if len(spread):
            between = ipsae_table["score_ipsae"].std()
            print(f"seed spread per design: median {spread.median():.3f}; "
                  f"between designs: sd {between:.3f}")
            print("Those two numbers are the honest context for whatever AUROC follows:")
            print("a column whose seed spread approaches its between-design spread is")
            print("ranking on noise, whatever it scores on a panel this size.")

In [12]:
# --- the headline: direction AUROC, overall, with its interval --------------
labels = direction["direction"].to_numpy(dtype=int)
scored_frame = direction.merge(
    direction_scored.frame.drop_duplicates("variant_id")[
        ["variant_id"] + [f"score_{m}" for m in direction_ran]
    ],
    on="variant_id", how="left",
)
if len(scored_frame) != len(direction):
    raise ValueError(f"the join changed the row count: {len(direction)} -> {len(scored_frame)}")

# ipSAE joins here rather than in the merge above, because it is not a column of
# the scored table: it comes from folds, keyed by the same variant ids.
if IPSAE_RAN:
    scored_frame["score_ipsae"] = scored_frame["variant_id"].map(IPSAE_SCORES)
    direction_ran = list(direction_ran) + ["ipsae"]

# Which negatives each scorer actually saw, before any number is read.
print(f"{'scorer':<14}{'kind':>8}{'pos':>6}{'neg':>6}{'AUROC':>8}"
      f"{'  ' + str(int(CI * 100)) + '% CI':>16}{'p':>10}{'how':>22}")
print("-" * 92)
overall = []
for model in direction_ran:
    column = f"score_{model}"
    values = scored_frame[column].to_numpy(dtype=float)
    point, n_pos, n_neg = auroc(values, labels)
    low, high = auroc_ci(values, labels)
    p, floor, how = permutation_p(values, labels)
    kind = "matrix" if model in MATRIX_MODELS else "multi"
    interval = "-" if low != low else f"{low:.3f} to {high:.3f}"
    print(f"{model:<14}{kind:>8}{n_pos:>6}{n_neg:>6}{cell(point):>8}{interval:>16}"
          f"{('-' if p != p else f'{p:.4f}'):>10}{how:>22}")
    overall.append({"model": model, "kind": kind, "n_pos": n_pos, "n_neg": n_neg,
                    "auroc": point, "ci_low": low, "ci_high": high, "p": p,
                    "p_floor": floor})
overall = pd.DataFrame(overall).set_index("model")
print("-" * 92)
print("AUROC 0.5 is a coin. An interval that contains 0.5 is a scorer this panel")
print("cannot distinguish from one, which with 21 against 8 is most of them.")
print("'pos' and 'neg' are what that scorer could answer, not the panel size: a")
print("matrix scorer sees the singles only, so its row is a different test.")

# The weak-negative sensitivity check, as promised in Part 1.
keep = scored_frame["label"] != WEAK_NEGATIVE
print(f"\nthe same table with {WEAK_NEGATIVE} dropped "
      f"({int((~keep).sum())} variant, the one the source calls 'similar'):")
print(f"{'scorer':<14}{'pos':>6}{'neg':>6}{'AUROC':>8}{'delta':>9}")
print("-" * 43)
for model in direction_ran:
    column = f"score_{model}"
    point, n_pos, n_neg = auroc(scored_frame.loc[keep, column].to_numpy(dtype=float),
                                labels[keep.to_numpy()])
    before = overall.loc[model, "auroc"]
    delta = point - before
    print(f"{model:<14}{n_pos:>6}{n_neg:>6}{cell(point):>8}"
          f"{('-' if delta != delta else f'{delta:+.3f}'):>9}")
print("-" * 43)
print("A scorer whose AUROC moves a lot here was being carried by one weak label.")

scorer            kind   pos   neg   AUROC          95% CI         p                   how
--------------------------------------------------------------------------------------------


antifold        matrix    10     3   0.700  0.400 to 0.967    0.1853        exact over 286


ablang2         matrix    10     3   0.667  0.000 to 1.000    0.2343        exact over 286


esm1v           matrix    10     3   0.233  0.000 to 0.600    0.9196        exact over 286


esm2_650m       matrix    10     3   0.400  0.000 to 0.800    0.7133        exact over 286


antiberty       matrix    10     3   0.367  0.000 to 0.767    0.7657        exact over 286


antifold_seq     multi    21     8   0.560  0.315 to 0.804    0.3285 sampled, 10,000 draws


foldx            multi    21     8   0.792  0.583 to 0.958    0.0083 sampled, 10,000 draws
--------------------------------------------------------------------------------------------
AUROC 0.5 is a coin. An interval that contains 0.5 is a scorer this panel
cannot distinguish from one, which with 21 against 8 is most of them.
'pos' and 'neg' are what that scorer could answer, not the panel size: a
matrix scorer sees the singles only, so its row is a different test.

the same table with L:R66G dropped (1 variant, the one the source calls 'similar'):
scorer           pos   neg   AUROC    delta
-------------------------------------------
antifold          10     2   0.600   -0.100
ablang2           10     2   1.000   +0.333
esm1v             10     2   0.350   +0.117
esm2_650m         10     2   0.600   +0.200
antiberty         10     2   0.550   +0.183
antifold_seq      21     7   0.571   +0.012
foldx             21     7   0.762   -0.030
-------------------------------------------
A sco

In [13]:
# --- the split that matters: singles against multi-mutation -----------------
# The matrix scorers cannot appear in the multi column at all - they return an
# empty cell for a candidate with several substitutions, by design, because a
# lookup would have to add single scores. So the two columns are two different
# tests on two different sets and the gap between them is not a degradation
# measurement; it is a statement about which scorers were asked which question.
singles = ~scored_frame["multi"].to_numpy()
multi = scored_frame["multi"].to_numpy()

print(f"singles: {int(labels[singles].sum())} improved, "
      f"{int((1 - labels[singles]).sum())} worsened")
print(f"multi:   {int(labels[multi].sum())} improved, "
      f"{int((1 - labels[multi]).sum())} worsened")
print()
head = (f"{'scorer':<14}{'singles':>9}{'pos/neg':>10}{'  CI':>16}"
        f"{'multi':>9}{'pos/neg':>10}{'  CI':>16}")
print(head)
print("-" * len(head))
split = []
for model in direction_ran:
    column = f"score_{model}"
    row = {"model": model}
    line = f"{model:<14}"
    for mask in (singles, multi):
        values = scored_frame.loc[mask, column].to_numpy(dtype=float)
        point, n_pos, n_neg = auroc(values, labels[mask])
        low, high = auroc_ci(values, labels[mask])
        interval = "-" if low != low else f"{low:.3f} to {high:.3f}"
        line += f"{cell(point):>9}{f'{n_pos}/{n_neg}':>10}{interval:>16}"
        suffix = "singles" if mask is singles else "multi"
        row[f"auroc_{suffix}"] = point
        row[f"n_{suffix}"] = f"{n_pos}/{n_neg}"
    print(line)
    split.append(row)
split = pd.DataFrame(split).set_index("model")
print("-" * len(head))
n_singles, n_multi = int(singles.sum()), int(multi.sum())
singles_floor = 1.0 / comb(n_singles, int((1 - labels[singles]).sum()))
print(f"The singles column is {int(labels[singles].sum())} against "
      f"{int((1 - labels[singles]).sum())}, so its best achievable p is")
print(f"1/C({n_singles},{int((1 - labels[singles]).sum())}) = {singles_floor:.4f} - "
      f"better than the old test's {old_floor:.4f}, and still not a number")
print(f"anybody should act on. The multi column is {int(labels[multi].sum())} "
      f"against {int((1 - labels[multi]).sum())}.")

# --- by format, to see whether the errors cluster there ---------------------
print(f"\n{'scorer':<14}", end="")
for fmt in sorted(scored_frame["format"].unique()):
    print(f"{fmt:>22}", end="")
print()
print("-" * (14 + 22 * scored_frame["format"].nunique()))
by_format = []
for model in direction_ran:
    column = f"score_{model}"
    print(f"{model:<14}", end="")
    row = {"model": model}
    for fmt in sorted(scored_frame["format"].unique()):
        mask = (scored_frame["format"] == fmt).to_numpy()
        point, n_pos, n_neg = auroc(scored_frame.loc[mask, column].to_numpy(dtype=float),
                                    labels[mask])
        print(f"{cell(point) + f' ({n_pos}/{n_neg})':>22}", end="")
        row[fmt] = point
    print()
    by_format.append(row)
by_format = pd.DataFrame(by_format).set_index("model")
print("-" * (14 + 22 * scored_frame["format"].nunique()))
print("'-' is a cell with too few of one label for an AUROC to exist, and Part 1")
print("said which cells those would be before anything was scored: scFv is")
print("negatives only and IgG is one positive against three. So the only format")
print("with a real AUROC is Fab, and the pooled number above is mostly it.")
print("That is a limit of the panel, not a finding about formats: this set cannot")
print("tell you whether a scorer transfers from Fab to IgG.")

for name, frame in (("overall", overall), ("by_class", split), ("by_format", by_format)):
    path = out_dir / "direction" / f"auroc_{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path)
    print(f"-> {path}")

singles: 10 improved, 3 worsened
multi:   11 improved, 5 worsened

scorer          singles   pos/neg              CI    multi   pos/neg              CI
------------------------------------------------------------------------------------


antifold          0.700      10/3  0.367 to 0.967        -       0/0               -


ablang2           0.667      10/3  0.000 to 1.000        -       0/0               -


esm1v             0.233      10/3  0.000 to 0.567        -       0/0               -


esm2_650m         0.400      10/3  0.000 to 0.800        -       0/0               -


antiberty         0.367      10/3  0.000 to 0.767        -       0/0               -


antifold_seq      0.600      10/3  0.300 to 0.900    0.564      11/5  0.236 to 0.873


foldx             0.733      10/3  0.200 to 1.000    0.873      11/5  0.655 to 1.000
------------------------------------------------------------------------------------
The singles column is 10 against 3, so its best achievable p is
1/C(13,3) = 0.0035 - better than the old test's 0.0182, and still not a number
anybody should act on. The multi column is 11 against 5.

scorer                           Fab                   IgG                  scFv
--------------------------------------------------------------------------------
antifold                 0.556 (9/2)               - (1/0)               - (0/1)
ablang2                  1.000 (9/2)               - (1/0)               - (0/1)
esm1v                    0.278 (9/2)               - (1/0)               - (0/1)
esm2_650m                0.556 (9/2)               - (1/0)               - (0/1)
antiberty                0.500 (9/2)               - (1/0)               - (0/1)
antifold_seq            0.200 (20/2)               - (1/3)    

## Part 3 - the Chinery binary: clean kinetics, not weak affinity

The Chinery 2026 block is the one place in the evidence set with enough labelled
candidates to make a real AUROC, and its label is **not** the Part 2 label. Read
this before the numbers, because the difference decides what the AUROC means.

720 CDR-H3 redesigns were submitted as IgG on a Carterra LSA. The negatives file
labels what came back:

| label | source value | wells | what it means |
|---|---|---|---|
| `positive` | `fitted_KD` | 156 | a clean 1:1 fit and a numeric KD |
| `negative_clean` | `bound_unfittable` | 97 | `I.C.` - binding was seen and could not be fit reliably |
| `negative_caveated` | `source_NA` | 434 | `N/A` - no fit at all |

### What the 97 actually are, decided from the columns

It would be easy to read `bound_unfittable` as "bound weakly". The evidence CSV
carries the instrument's own per-well QC, so the cell below decides it from
those columns instead of from the paper's prose, and the answer is not weakness:

| column | 156 fitted | 97 unfittable | test |
|---|---|---|---|
| `source_capture_RU` | 247.2 (IQR 222-270) | 251.2 (226-274) | p 0.43, AUC 0.47 |
| `source_yield_ug` | 464.9 (413-529) | 482.2 (403-524) | p 0.74, AUC 0.49 |
| `source_purity_percent` | 100.0 | 100.0 | p 0.80 |
| `source_quant_status` | Pass x156 | Pass x97 | identical |
| `source_fit_residual_sd` | numeric, median 3.60 | **`I.C.` x97** | no magnitude to compare |

Capture RU is how much antibody the sensor held, and it is the same in both
groups; so are yield, purity and the quantitation status. Expression failure is
ruled out three ways over, and there was just as much material in the
unfittable wells. The single column that differs is the fit residual, and it
differs categorically rather than by degree: the source declines to fit, writing
`I.C.` with `failure_category = "inconclusive kinetic fit"` and
`outcome = "binding observed; unreliable 1:1 fit"`.

**So the 97 are a kinetic-heterogeneity label - a developability failure - and
not a weak-affinity label.** What a scorer that separates them has shown is that
it can tell a design that binds cleanly 1:1 from one that binds with
heterogeneous kinetics (avidity, conformational exchange, a sticky or partly
unfolded interface). That is worth having, and it is a different claim from
Part 2's. Nothing in this notebook pools the two, and Part 5 keeps them in
separate columns for exactly this reason: **an AUROC here is not evidence about
affinity direction**, however large its n.

It is also a mechanistically plausible thing for a stability-based scorer to be
good at, which is a reason to be careful rather than pleased: FoldX penalises a
destabilised interface, and a destabilised interface is what produces non-1:1
kinetics. A high AUROC here is consistent with FoldX measuring interface
integrity and saying nothing about affinity.

`negative_caveated` is `N/A`, and **`N/A` is ambiguous in the source.** It is
consistent with a genuine non-binder, and equally consistent with a well that
failed to express, a sensor that did not regenerate, or a design that never
reached the chip. The evidence CSV is explicit that it does not convert `N/A`
into a demonstrated non-binder. Including those 434 adds 4.5x the negatives and
an unknown amount of label noise pointing in an unknown direction, so Part 3
reports **both** tables and neither as the answer.

**The base rate falls with substitution count, which is why the within-size
table exists.** Among the clean split it runs from about 70% positive at four
substitutions to 29% at ten. A pooled AUROC over that is partly measuring size,
exactly as the pooled Spearman in `03_chinery_size_calibration.ipynb` was, and a
within-size AUROC is read against the right null because inside a size bin the
base rate is fixed. The base rate per bin is printed beside every AUROC for the
same reason.

**And the whole block is selected on outcome.** The 156 are the designs that
produced a fit out of 720 submitted; the KD range is truncated and the
truncation is size-dependent. Every correlation in that notebook is attenuated
by it and so is every AUROC here. Each is a lower bound.

In [14]:
# --- the Chinery block, one row per well ------------------------------------
chinery_rows = negatives_file[negatives_file["study"] == CHINERY_STUDY]
wanted = (CHINERY_POSITIVE, CHINERY_CLEAN, CHINERY_CAVEATED)
chinery_rows = chinery_rows[chinery_rows["label_class"].isin(wanted)]

records = []
for (well, label_class), group in chinery_rows.groupby(["variant", "label_class"]):
    mutations = [
        (str(r.chain), int(r.position), str(r.wt), str(r.mut)) for r in group.itertuples()
    ]
    declared = sorted(set(int(n) for n in group["n_mutations"]))
    if declared != [len(mutations)]:
        raise ValueError(f"{well}: {len(mutations)} rows but n_mutations says {declared}")
    records.append({
        "well": well, "label_class": label_class,
        "label": label_of(mutations), "n_mutations": len(mutations),
        "reason": sorted(set(group["label_reason"]))[0],
        "mutations": tuple(sorted(remap(mutations))),
    })
chinery = pd.DataFrame(records)
chinery["variant_id"] = [make_id(m) for m in chinery["mutations"]]
chinery["binary"] = (chinery["label_class"] == CHINERY_POSITIVE).astype(int)

counts = chinery.groupby("label_class")["well"].nunique().to_dict()
print(f"{CHINERY_STUDY}: {len(chinery)} wells over {chinery['variant_id'].nunique()} "
      f"distinct substitution sets")
for label_class in wanted:
    print(f"  {label_class:<20}{counts.get(label_class, 0):>5}  "
          f"({sorted(set(chinery.loc[chinery['label_class'] == label_class, 'reason']))})")
if counts != CHINERY_EXPECTED:
    raise ValueError(f"expected {CHINERY_EXPECTED}, got {counts}")

if chinery["variant_id"].duplicated().any():
    dupes = chinery.loc[chinery["variant_id"].duplicated(keep=False)].sort_values("variant_id")
    print(f"\n{dupes['variant_id'].nunique()} substitution sets appear in more than one "
          f"well; a scorer gives one number per set, so the duplicates would be scored "
          f"once and counted twice:")
    print(dupes[["well", "label_class", "n_mutations"]].to_string(index=False))
    raise ValueError(
        "two wells share a substitution set. Decide what the label of that set is "
        "before scoring it - if the two wells disagree, that is the assay's "
        "reproducibility and it belongs in the caveats, not averaged away here."
    )

# --- reconcile with the evidence CSV's own census ---------------------------
# The two files were curated separately and they do not have to agree. Print the
# gap rather than assert it away: the evidence CSV counts every submitted row,
# the negatives file has already dropped the wells it could not express as a
# substitution set (the length-changing ablang designs, and anything with no
# substitution list at all).
submitted = evidence[(evidence["study"] == CHINERY_STUDY)
                     & (evidence["format"] == "IgG")
                     & (evidence["record_type"] == "variant_outcome")]

# `raw_KD_M` holds the literal strings "N/A" and "I.C.", and "N/A" is in pandas'
# default list of missing-value markers - so in the frame read at the top of
# Part 1 every N/A well is already a NaN and `== "N/A"` is False everywhere.
# That is a silent miscount, not an error, so the column is re-read here with
# the default NA markers switched off and nothing else changed.
raw_status = pd.read_csv(
    EVIDENCE, low_memory=False, keep_default_na=False, na_values=[],
    usecols=["study", "format", "record_type", "raw_KD_M"],
)
raw_status = raw_status[(raw_status["study"] == CHINERY_STUDY)
                       & (raw_status["format"] == "IgG")
                       & (raw_status["record_type"] == "variant_outcome")]
if len(raw_status) != len(submitted):
    raise ValueError(f"the re-read selects {len(raw_status)} rows, not {len(submitted)}")
status = raw_status["raw_KD_M"].astype(str).str.strip()
census = {
    "numeric KD (a clean 1:1 fit)": int((submitted["metric"] == "KD").sum()),
    "I.C. (binding seen, not reliably fit)": int((status == "I.C.").sum()),
    "N/A (no fit)": int((status == "N/A").sum()),
    "blank": int((status == "").sum()),
}
if sum(census.values()) != len(submitted):
    raise ValueError(f"the census accounts for {sum(census.values())} of "
                     f"{len(submitted)} submitted designs")
print(f"\n{Path(EVIDENCE).name} census of the same block "
      f"({len(submitted)} submitted designs):")
pairs = ((CHINERY_POSITIVE, "numeric KD (a clean 1:1 fit)"),
         (CHINERY_CLEAN, "I.C. (binding seen, not reliably fit)"),
         (CHINERY_CAVEATED, "N/A (no fit)"))
print(f"{'label':<22}{'negatives file':>16}{'evidence file':>15}{'gap':>6}")
print("-" * 59)
for label_class, key in pairs:
    here, there = counts.get(label_class, 0), census[key]
    print(f"{label_class:<22}{here:>16}{there:>15}{there - here:>6}")
print(f"{'(not labelled)':<22}{'-':>16}{census['blank']:>15}{'':>6}")
print("-" * 59)
print(f"{'total':<22}{sum(counts.values()):>16}{len(submitted):>15}"
      f"{len(submitted) - sum(counts.values()):>6}")
print()
print("The gaps are the wells the negatives file could not express as a")
print("substitution set: the length-changing designs, whose CDR-H3 is nine")
print("residues instead of ten, and the ones with no substitution list at all. A")
print("variant_id here is a list of substitutions and cannot express an indel, so")
print("they are absent from both halves of the binary and do not bias it - they do")
print(f"mean this is {sum(counts.values())} of {len(submitted)} and not all of it.")

# The panel's labels on disk, so anything scored outside this notebook - a fold,
# a tool with no adapter - can join on the same variant ids and the same binary
# rather than deriving a second version of the set from the benchmark CSVs.
chinery_path = out_dir / 'chinery' / 'chinery_set.csv'
chinery_path.parent.mkdir(parents=True, exist_ok=True)
chinery[['well', 'label_class', 'label', 'n_mutations', 'variant_id', 'binary']].to_csv(
    chinery_path, index=False
)
print()
print(f'-> {chinery_path}')


chinery2026: 687 wells over 687 distinct substitution sets
  positive              156  (['fitted_KD'])
  negative_clean         97  (['bound_unfittable'])
  negative_caveated     434  (['source_NA'])

trastuzumab_mutation_evidence.csv census of the same block (720 submitted designs):
label                   negatives file  evidence file   gap
-----------------------------------------------------------
positive                           156            159     3
negative_clean                      97            102     5
negative_caveated                  434            445    11
(not labelled)                       -             14      
-----------------------------------------------------------
total                              687            720    33

The gaps are the wells the negatives file could not express as a
substitution set: the length-changing designs, whose CDR-H3 is nine
residues instead of ten, and the ones with no substitution list at all. A
variant_id here is a list 

In [15]:
# --- what the 97 are, from the instrument's own columns ----------------------
# The label `bound_unfittable` does not say why the fit failed, and the three
# candidate explanations have different consequences for what Part 3's AUROC
# means: a weak binder is an affinity statement, a failed expression is an
# artefact, and kinetic heterogeneity is a developability statement. The
# evidence CSV carries the per-well QC, so this decides it rather than quoting
# the paper.
from scipy.stats import mannwhitneyu

QC_COLUMNS = ["source_capture_RU", "source_fit_residual_sd",
              "source_yield_ug", "source_purity_percent"]

evidence_all = pd.read_csv(EVIDENCE, low_memory=False, dtype=str)
chinery_evidence = evidence_all[evidence_all["study"] == CHINERY_STUDY]
labelled = (negatives_file[negatives_file["study"] == CHINERY_STUDY]
            .drop_duplicates("variant")[["variant", "label_class"]])
provenance = labelled.merge(chinery_evidence, on="variant", how="left")
provenance = provenance[provenance["label_class"].isin([CHINERY_POSITIVE, CHINERY_CLEAN])]
if len(provenance) != CHINERY_EXPECTED["positive"] + CHINERY_EXPECTED["negative_clean"]:
    raise ValueError(f"{len(provenance)} wells joined to the evidence file, expected "
                     f"{CHINERY_EXPECTED['positive'] + CHINERY_EXPECTED['negative_clean']}")

print(f"{'column':<24}{'group':<16}{'n':>5}{'median':>10}{'q25':>10}{'q75':>10}"
      f"   comparison")
print("-" * 92)
provenance_rows = []
for column in QC_COLUMNS:
    numeric = {}
    for label in (CHINERY_POSITIVE, CHINERY_CLEAN):
        values = pd.to_numeric(
            provenance.loc[provenance["label_class"] == label, column], errors="coerce"
        ).dropna()
        numeric[label] = values
        raw = provenance.loc[provenance["label_class"] == label, column]
        shown = (f"{values.median():>10.2f}{values.quantile(.25):>10.2f}"
                 f"{values.quantile(.75):>10.2f}" if len(values)
                 else f"{raw.value_counts().index[0] if len(raw) else '-':>10}{'':>20}")
        print(f"{column:<24}{label:<16}{len(raw):>5}{shown}")
    a, b = numeric[CHINERY_POSITIVE], numeric[CHINERY_CLEAN]
    if len(a) > 2 and len(b) > 2:
        statistic, p = mannwhitneyu(a, b)
        auc = statistic / (len(a) * len(b))
        verdict = "indistinguishable" if p > 0.05 else "DIFFERS"
        print(f"{'':<24}{'->':<16}{'':>5}{'':>30}   Mann-Whitney p {p:.3f}, "
              f"AUC {auc:.3f}  {verdict}")
        provenance_rows.append({"column": column, "p": p, "auc": auc,
                                "n_fitted": len(a), "n_unfittable": len(b)})
    else:
        print(f"{'':<24}{'->':<16}{'':>5}{'':>30}   not comparable: the "
              f"unfittable group carries no number here")
        provenance_rows.append({"column": column, "p": float("nan"),
                                "auc": float("nan"), "n_fitted": len(a),
                                "n_unfittable": len(b)})
print("-" * 92)
for column in ("source_quant_status", "outcome", "failure_category"):
    if column in provenance.columns:
        print(f"\n{column}:")
        print(pd.crosstab(provenance[column].fillna("(blank)"),
                          provenance["label_class"]).to_string())

provenance_table = pd.DataFrame(provenance_rows)
(out_dir / "chinery").mkdir(parents=True, exist_ok=True)
provenance_table.to_csv(out_dir / "chinery" / "label_provenance.csv", index=False)
print(f"\n-> {out_dir / 'chinery' / 'label_provenance.csv'}")
print()
print("Reading of the above: capture RU, yield, purity and quantitation status are")
print("statistically indistinguishable between the two groups, so neither expression")
print("nor the amount of material on the sensor explains the split, and the source")
print("records binding in both. The only column that separates them is the fit")
print("residual, which the source refuses to give a number for at all. That is")
print("kinetic heterogeneity, which is a DEVELOPABILITY label - so Part 3 measures")
print("'can this scorer spot a design that binds cleanly 1:1', and never affinity.")

column                  group               n    median       q25       q75   comparison
--------------------------------------------------------------------------------------------
source_capture_RU       positive          156    247.20    221.95    270.27
source_capture_RU       negative_clean     97    251.20    226.30    274.00
                        ->                                                    Mann-Whitney p 0.428, AUC 0.470  indistinguishable
source_fit_residual_sd  positive          156      3.60      2.68      5.03
source_fit_residual_sd  negative_clean     97      I.C.                    
                        ->                                                    not comparable: the unfittable group carries no number here
source_yield_ug         positive          156    464.94    412.66    528.90
source_yield_ug         negative_clean     97    482.16    403.44    523.98
                        ->                                                    Mann-Whitney p 0.

In [16]:
# --- the base rate per substitution count, before any scorer is involved ----
def base_rates(frame, negative_classes):
    rows = []
    subset = frame[frame["label_class"].isin((CHINERY_POSITIVE,) + tuple(negative_classes))]
    for size in sorted(subset["n_mutations"].unique()):
        at_size = subset[subset["n_mutations"] == size]
        n_pos = int((at_size["binary"] == 1).sum())
        n_neg = int((at_size["binary"] == 0).sum())
        rows.append({"n_sub": int(size), "positive": n_pos, "negative": n_neg,
                     "total": n_pos + n_neg,
                     "base_rate": n_pos / float(n_pos + n_neg) if n_pos + n_neg else float("nan")})
    table = pd.DataFrame(rows).set_index("n_sub")
    return table


clean_rates = base_rates(chinery, (CHINERY_CLEAN,))
caveated_rates = base_rates(chinery, (CHINERY_CLEAN, CHINERY_CAVEATED))

print("base rate = share of wells at this size that returned a numeric KD")
print(f"{'n_sub':>6}{'pos':>6}{'neg':>6}{'total':>7}{'base rate':>11}"
      f"{'  |':>4}{'pos':>6}{'neg':>6}{'total':>7}{'base rate':>11}")
print(f"{'':>6}{'clean split (positive vs negative_clean)':^30}{'':>4}"
      f"{'with negative_caveated':^30}")
print("-" * 76)
for size in sorted(set(clean_rates.index) | set(caveated_rates.index)):
    line = f"{size:>6}"
    for table in (clean_rates, caveated_rates):
        if size in table.index:
            row = table.loc[size]
            line += (f"{int(row['positive']):>6}{int(row['negative']):>6}"
                     f"{int(row['total']):>7}{row['base_rate']:>10.1%}")
        else:
            line += f"{'-':>6}{'-':>6}{'-':>7}{'-':>11}"
        line += f"{'  |':>4}" if table is clean_rates else ""
    print(line)
print("-" * 76)
for name, table in (("clean", clean_rates), ("with caveated", caveated_rates)):
    pooled = table["positive"].sum() / float(table["total"].sum())
    trend = metrics.spearman(table.index.to_numpy(dtype=float),
                             table["base_rate"].to_numpy(dtype=float))
    print(f"{name:<15} pooled base rate {pooled:.1%}, "
          f"Spearman of base rate against size {trend:+.3f}")
print()
print("The base rate falls with size, so a pooled AUROC over the whole block is")
print("partly a size detector - the same confound the pooled Spearman had in")
print("03_chinery_size_calibration.ipynb. Inside a size bin the base rate is")
print("fixed, which is the only place 0.5 is the right null.")
print("Sizes 1 and 2 have 0 and 1 clean negatives, so CHINERY_SIZES starts at 3.")

base rate = share of wells at this size that returned a numeric KD
 n_sub   pos   neg  total  base rate   |   pos   neg  total  base rate
      clean split (positive vs negative_clean)        with negative_caveated    
----------------------------------------------------------------------------
     1    14     0     14    100.0%   |    14     0     14    100.0%
     2    13     1     14     92.9%   |    13     2     15     86.7%
     3    17     9     26     65.4%   |    17    24     41     41.5%
     4    32    14     46     69.6%   |    32    38     70     45.7%
     5    27    11     38     71.1%   |    27    51     78     34.6%
     6    23    16     39     59.0%   |    23    80    103     22.3%
     7    14    13     27     51.9%   |    14    87    101     13.9%
     8     8    12     20     40.0%   |     8    94    102      7.8%
     9     4    11     15     26.7%   |     4    92     96      4.2%
    10     4    10     14     28.6%   |     4    63     67      6.0%
--------------

In [17]:
# --- score the Chinery block ------------------------------------------------
# 253 candidates for the clean split, 687 with the caveated negatives. For the
# per-variant scorers that is the whole cost of this notebook: FoldX is
# FOLDX_RUNS BuildModel runs plus two AnalyseComplex calls per candidate, so
# point DESIGN_PIPELINE_FOLDX_PDB at the trimmed, repaired Fv plus antigen
# domain the alanine scan used or it is days rather than hours.
classes = [CHINERY_POSITIVE, CHINERY_CLEAN]
if SCORE_CAVEATED:
    classes.append(CHINERY_CAVEATED)
to_score = chinery[chinery["label_class"].isin(classes)].reset_index(drop=True)
print(f"scoring {len(to_score)} candidates "
      f"({'clean split plus the caveated negatives' if SCORE_CAVEATED else 'the clean split only'})")
if not SCORE_CAVEATED:
    print("SCORE_CAVEATED is False, so the second table below is reported as not computed.")
if len(to_score) > MAX_VARIANTS:
    raise ValueError(
        f"{len(to_score)} candidates against MAX_VARIANTS={MAX_VARIANTS}. That cap is "
        f"what stops an uncapped per-variant scorer queueing a week of compute; raise "
        f"it deliberately, or set SCORE_CAVEATED = False."
    )

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="anarci is not installed")
    chinery_table = enumerate_variants(
        complex_, HEAVY_CHAIN, out_dir / "chinery" / "variants",
        only=to_score["variant_id"].tolist(), structure=pdb,
    )
print(f"{chinery_table.rows} rows, {chinery_table.candidates} candidates")
if chinery_table.candidates != len(to_score):
    raise ValueError(f"{chinery_table.candidates} candidates for {len(to_score)} wells")

print("\nscoring the Chinery block:")
chinery_scored, chinery_ran, chinery_skipped = score_set(
    chinery_table, ALL_MODELS, out_dir / "chinery" / "scores", "Chinery block"
)
chinery_panel = to_score.merge(
    chinery_scored.frame.drop_duplicates("variant_id")[
        ["variant_id"] + [f"score_{m}" for m in chinery_ran]
    ],
    on="variant_id", how="left",
)
if len(chinery_panel) != len(to_score):
    raise ValueError("the join changed the row count")
print(f"\nNote which scorers are absent here and why: every Chinery design carries")
print(f"at least one substitution and {int((chinery_panel['n_mutations'] > 1).sum())} of "
      f"{len(chinery_panel)} carry several, so the")
print("matrix scorers answer only the singles - 14 of 687 - and their rows below")
print("are thin by construction rather than by accident.")

scoring 253 candidates (the clean split only)
SCORE_CAVEATED is False, so the second table below is reported as not computed.
1352 rows, 253 candidates

scoring the Chinery block:
  antifold          14 of 253 candidates scored
  ablang2           14 of 253 candidates scored
  esm1v             14 of 253 candidates scored
  esm2_650m         14 of 253 candidates scored
  antiberty         14 of 253 candidates scored
  consensus      SKIPPED  MissingDependencyError: the consensus scorer needs an OAS frequency table and none is configured. Pass frequencies=<path>, or set DESIGN_PIPELINE_OAS_TABLE. The file is a CSV of imgt,aa,frequency (or position,aa,frequency), built once from the Observed Antibody Space download at https://opig.stats.ox.ac.uk/webapps/oas/. Drop 'consensus' from models=[...] to run without it.


  antifold_seq     253 of 253 candidates scored
  foldx            253 of 253 candidates scored
  gearbind       SKIPPED  MissingRuntimeError: gearbind is not installed. Verified route (checked 2026-10-30) [container]: build an image from https://github.com/DeepGraphLearning/GearBind and set it with REGISTRY['gearbind'] - no published image exists. Note: TorchDrug pins Python 3.7-3.10 and an old torch: the one genuine conflict in this set, hence a container rather than Env B.. `python -m design_pipeline.preflight` prints the whole table.
Chinery block: scorers that ran: antifold, ablang2, esm1v, esm2_650m, antiberty, antifold_seq, foldx

Note which scorers are absent here and why: every Chinery design carries
at least one substitution and 239 of 253 carry several, so the
matrix scorers answer only the singles - 14 of 687 - and their rows below
are thin by construction rather than by accident.


In [18]:
# --- the Chinery AUROC: overall, then within each substitution count --------
def chinery_auroc_table(negative_classes, title):
    subset = chinery_panel[
        chinery_panel["label_class"].isin((CHINERY_POSITIVE,) + tuple(negative_classes))
    ]
    y = subset["binary"].to_numpy(dtype=int)
    print(f"\n{title}")
    print(f"  {int((y == 1).sum())} positive, {int((y == 0).sum())} negative, "
          f"pooled base rate {y.mean():.1%}")
    head = (f"{'scorer':<14}{'pooled':>8}{'pos/neg':>10}{'  CI':>16}"
            + "".join(f"{f'n={s}':>9}" for s in CHINERY_SIZES))
    print(head)
    print("  " + "-" * (len(head) - 2))
    rows = []
    for model in chinery_ran:
        column = f"score_{model}"
        values = subset[column].to_numpy(dtype=float)
        point, n_pos, n_neg = auroc(values, y)
        low, high = auroc_ci(values, y)
        interval = "-" if low != low else f"{low:.3f} to {high:.3f}"
        line = f"{model:<14}{cell(point):>8}{f'{n_pos}/{n_neg}':>10}{interval:>16}"
        row = {"model": model, "auroc": point, "ci_low": low, "ci_high": high,
               "n_pos": n_pos, "n_neg": n_neg}
        for size in CHINERY_SIZES:
            mask = (subset["n_mutations"] == size).to_numpy()
            at_size, _, _ = auroc(subset.loc[mask, column].to_numpy(dtype=float), y[mask])
            line += f"{cell(at_size):>9}"
            row[f"auroc_n{size}"] = at_size
        print(line)
        rows.append(row)
    print("  " + "-" * (len(head) - 2))
    line = f"{'base rate':<14}{y.mean():>8.3f}{'':>10}{'':>16}"
    for size in CHINERY_SIZES:
        mask = (subset["n_mutations"] == size).to_numpy()
        line += f"{(y[mask].mean() if mask.any() else float('nan')):>9.3f}"
    print(line)
    print("  the base-rate row is not an AUROC - it is what share of that bin is")
    print("  positive, printed so each column is read against its own population.")
    print("  AUROC's null is 0.5 whatever the base rate; the base rate is what tells")
    print("  you whether a bin has enough of both labels to mean anything.")
    return pd.DataFrame(rows).set_index("model")


chinery_clean_auroc = chinery_auroc_table(
    (CHINERY_CLEAN,), "clean split: positive (fitted KD) against negative_clean (I.C.)"
)
if SCORE_CAVEATED:
    chinery_all_auroc = chinery_auroc_table(
        (CHINERY_CLEAN, CHINERY_CAVEATED),
        "with negative_caveated included: positive against I.C. plus N/A",
    )
    print("\nThe two tables answer different questions and the second one's negatives")
    print("are the ambiguous ones. 'N/A' in the source is consistent with a genuine")
    print("non-binder and equally consistent with a well that failed to express or a")
    print("sensor that did not regenerate; the evidence CSV refuses to call it a")
    print("demonstrated non-binder and so does this notebook. Where a scorer looks")
    print("better in the second table, the honest reading is that it separates")
    print("'something went wrong with this well' from 'this bound', which is not the")
    print("claim anybody wants to make.")
else:
    chinery_all_auroc = None
    print("\nSCORE_CAVEATED is False, so the caveated table was not computed. It is")
    print("not reported as 'no difference' - it was not run.")

for name, frame in (("clean", chinery_clean_auroc), ("with_caveated", chinery_all_auroc)):
    if frame is None:
        continue
    path = out_dir / "chinery" / f"auroc_{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path)
    print(f"-> {path}")
print()
print("And the caveat that attenuates both: the block is selected on outcome -")
print("156 fits out of 720 submitted - and the selection is size-dependent, so")
print("every AUROC here is a lower bound and the right-hand size columns are the")
print("survivors of a harsher filter than the left-hand ones.")


clean split: positive (fitted KD) against negative_clean (I.C.)
  156 positive, 97 negative, pooled base rate 61.7%
scorer          pooled   pos/neg              CI      n=3      n=4      n=5      n=6      n=7      n=8      n=9     n=10
  ----------------------------------------------------------------------------------------------------------------------
antifold             -      14/0               -        -        -        -        -        -        -        -        -
ablang2              -      14/0               -        -        -        -        -        -        -        -        -
esm1v                -      14/0               -        -        -        -        -        -        -        -        -
esm2_650m            -      14/0               -        -        -        -        -        -        -        -        -
antiberty            -      14/0               -        -        -        -        -        -        -        -        -


antifold_seq     0.610    156/97  0.537 to 0.679    0.673    0.654    0.465    0.454    0.308    0.292    0.250    0.575


foldx            0.716    156/97  0.651 to 0.780    0.686    0.728    0.620    0.685    0.692    0.656    0.000    0.500
  ----------------------------------------------------------------------------------------------------------------------
base rate        0.617                              0.654    0.696    0.711    0.590    0.519    0.400    0.267    0.286
  the base-rate row is not an AUROC - it is what share of that bin is
  positive, printed so each column is read against its own population.
  AUROC's null is 0.5 whatever the base rate; the base rate is what tells
  you whether a bin has enough of both labels to mean anything.

SCORE_CAVEATED is False, so the caveated table was not computed. It is
not reported as 'no difference' - it was not run.
-> runs/negative_controls/chinery/auroc_clean.csv

And the caveat that attenuates both: the block is selected on outcome -
156 fits out of 720 submitted - and the selection is size-dependent, so
every AUROC here is a lower bound and the

## Part 4 - manufactured negatives, as a floor check

Real negatives are scarce: Part 2 has eight of them and three share a
substitution. Confident negatives, on the other hand, are free - you can
construct a variant that no reasonable model should like and check whether the
scorer likes it. That is a weaker test than a measurement and it is a *different*
test, because it has no false-negative risk at all: nobody needs an SPR run to
know that putting a proline in the middle of a beta strand is bad.

**This part cannot tell you a scorer is good. It can tell you a scorer is
broken, and a scorer that fails here fails regardless of what Parts 2 and 3
said.** A scorer with AUROC 0.7 on eight negatives and a 60% failure rate on
manufactured ones has learned something other than binding.

Five classes, four of them deliberate breaks and one a control on the control:

* **Alanine at each of the ten FoldX hotspots.** `design_pipeline.hotspots`
  holds them: positions that lose at least 1.0 kcal/mol of binding on alanine,
  all within 3.8 A of HER2. Alanine removes the side chain past its first
  carbon, so these are the positions whose side chains carry the interaction.
  **FoldX is not scored on these** - the hotspot list *is* FoldX output, so
  asking FoldX to rank it is asking FoldX whether it agrees with itself.
  AntiFold, GearBind and the language models never saw it.
* **Physically implausible substitutions**, derived from the coordinates rather
  than from a list: a proline in the middle of a beta strand (proline has no
  amide hydrogen and the wrong phi, so it breaks the strand's hydrogen-bond
  ladder), a glycine replacing a buried bulky side chain (a cavity where a
  packed core used to be), and a charge reversal at a salt bridge (an attraction
  turned into a repulsion). None of these three comes from FoldX, so every
  scorer including FoldX is graded on them.
* **Non-interface surface substitutions, as neutral decoys.** Alanine at exposed
  positions outside the paratope. These are the control on the control: they
  **should score near zero**. A scorer that hands them large positive numbers is
  not distinguishing the interface from the bulk of the protein, and its
  enthusiasm for a real design is then enthusiasm about something else. FoldX is
  not scored on these either, for the same circularity reason as the hotspots.

**The sign convention is what makes a single threshold meaningful.**
`score_<model>` is higher-is-better-binding for every scorer in the registry,
and zero is the wild type *by construction* in every case: the matrix scorers
and `antifold_seq` report a log-likelihood ratio against wild type, FoldX and
GearBind report a negated ddG. So "scored above zero" means "the scorer calls
this deliberate break an improvement", and that is the failure rate reported
below. The harder criterion beside it is "scored above the median of the real
positives from Part 1", which is what a shortlist cut would actually act on.

And one more control that needs no manufactured variant at all, in the cell
after: `wt_argmax_rate`, which turns every interface position into its own
control.

In [19]:
# --- the structure, read once for burial, secondary structure and contacts --
import biotite.structure as struc

# Theoretical maximum solvent-accessible side-chain area per residue, from
# Tien et al. 2013 (PLoS ONE 8:e80635, Table 1, theoretical values). Relative
# SASA is the residue's area over this, which is the usual definition of burial
# and is what BURIED_BELOW and SURFACE_ABOVE are thresholds on.
MAX_ASA = {
    "A": 129.0, "R": 274.0, "N": 195.0, "D": 193.0, "C": 167.0, "E": 223.0,
    "Q": 225.0, "G": 104.0, "H": 224.0, "I": 197.0, "L": 201.0, "K": 236.0,
    "M": 224.0, "F": 240.0, "P": 159.0, "S": 155.0, "T": 172.0, "W": 285.0,
    "Y": 263.0, "V": 174.0,
}

atoms = structures.load(pdb)
atoms = atoms[struc.filter_amino_acids(atoms)]
antibody = atoms[np.isin(atoms.chain_id, [HEAVY_CHAIN, LIGHT_CHAIN])]

# Burial and secondary structure are computed on the antibody alone, with HER2
# removed. That is deliberate: a buried core position should be buried by its own
# domain, not by the antigen, and a position buried only in the complex is an
# interface position - which the paratope set below already names.
#
# They are computed over the *whole* of both chains, because that is the real
# molecule and a residue's burial depends on what is packed against it. The
# variable-domain restriction is applied when the classes are built, not here:
# 1N8Z carries a whole light chain and a heavy Fd, so most of each chain is
# constant domain that this pipeline never designs in.
atom_sasa = struc.sasa(antibody, vdw_radii="Single")
residue_sasa = struc.apply_residue_wise(antibody, atom_sasa, np.nansum)
starts = struc.get_residue_starts(antibody)
residue_chain = antibody.chain_id[starts]

# Atom index -> residue index, which is the map the salt-bridge search needs:
# `starts` holds atom indices and every per-residue array below is indexed by
# residue. Conflating the two is the bug this exists to make impossible.
bounds = list(starts) + [antibody.array_length()]
residue_of_atom = np.empty(antibody.array_length(), dtype=int)
for residue, (begin, end) in enumerate(zip(bounds, bounds[1:])):
    residue_of_atom[begin:end] = residue

relative_sasa, sse_of, residue_at = {}, {}, {}
for chain in (HEAVY_CHAIN, LIGHT_CHAIN):
    picked = np.flatnonzero(residue_chain == chain)
    sequence = sequences[chain]
    if len(picked) != len(sequence):
        raise IndexMismatchError(
            f"chain {chain}: {len(picked)} residues in the SASA pass against "
            f"{len(sequence)} in chain_sequence. The two disagree about what a "
            f"residue is, so positions cannot be mapped between them."
        )
    for position, index in enumerate(picked, start=1):
        residue_at[int(index)] = (chain, position)
        reference = MAX_ASA.get(sequence[position - 1])
        relative_sasa[(chain, position)] = (
            float(residue_sasa[index]) / reference if reference else float("nan")
        )
    sub = antibody[antibody.chain_id == chain]
    sse = struc.annotate_sse(sub)
    if len(sse) != len(sequence):
        raise IndexMismatchError(
            f"chain {chain}: annotate_sse returned {len(sse)} assignments for "
            f"{len(sequence)} residues, so they cannot be lined up by order."
        )
    # biotite 2.0 returns PSeaElement integers, not letters, and to_symbols is
    # the documented way back to 'a' / 'b' / 'c'. Converted here rather than
    # compared against a hard-coded 2, which is the kind of thing that silently
    # becomes wrong when the enum gains a member.
    symbols = (struc.PSeaElement.to_symbols(sse)
               if hasattr(struc, "PSeaElement") else sse)
    for position, code in enumerate(symbols, start=1):
        sse_of[(chain, position)] = str(code)


def in_variable_domain(chain, position):
    return position <= VARIABLE_DOMAIN.get(chain, 0)


paratope = metrics.interface_residues(
    pdb, [HEAVY_CHAIN, LIGHT_CHAIN], [ANTIGEN_CHAIN], cutoff=PARATOPE_CUTOFF
)
outside = sorted(k for k in paratope if not in_variable_domain(*k))
print(f"paratope at {PARATOPE_CUTOFF} A: {len(paratope)} positions "
      f"({sum(1 for c, _ in paratope if c == HEAVY_CHAIN)} on {HEAVY_CHAIN}, "
      f"{sum(1 for c, _ in paratope if c == LIGHT_CHAIN)} on {LIGHT_CHAIN})")
if outside:
    print(f"  {len(outside)} of them are outside the variable domains: {outside}")
    print("  which would mean HER2 is contacting a constant domain in this")
    print("  structure - worth a look before trusting the decoy set, since the")
    print("  decoys are defined as 'not in the paratope'.")

print(f"\nchains {HEAVY_CHAIN} and {LIGHT_CHAIN}: {len(relative_sasa)} residues, "
      f"{sum(1 for k in relative_sasa if in_variable_domain(*k))} of them in the "
      f"variable domains (VH 1-{VARIABLE_DOMAIN[HEAVY_CHAIN]}, "
      f"VL 1-{VARIABLE_DOMAIN[LIGHT_CHAIN]})")
counts = pd.Series([v for k, v in sse_of.items() if in_variable_domain(*k)]).value_counts()
print(f"secondary structure over the variable domains (biotite P-SEA): "
      + ", ".join(f"{k}={v}" for k, v in counts.items())
      + "  (a = helix, b = strand, c = coil)")
buried = [k for k, v in relative_sasa.items()
          if v < BURIED_BELOW and in_variable_domain(*k)]
surface = [k for k, v in relative_sasa.items()
           if v > SURFACE_ABOVE and in_variable_domain(*k)]
print(f"relative SASA in the variable domains: {len(buried)} positions under "
      f"{BURIED_BELOW} (buried), {len(surface)} over {SURFACE_ABOVE} (exposed)")

# --- IMGT regions, for the position-matched control -------------------------
# Needed to show that the between-position nearmiss comparison cannot be
# matched on this molecule: CDR membership and distance to the antigen are
# collinear among the acidic positions. Empty where ANARCI is absent, which
# that cell reports rather than guessing a CDR boundary.
IMGT_CDR_RANGES = {"CDR1": (27, 38), "CDR2": (56, 65), "CDR3": (105, 117)}
imgt_region_of, imgt_label_of = {}, {}
if numbering.available():
    for chain in (HEAVY_CHAIN, LIGHT_CHAIN):
        for index, label in enumerate(numbering.imgt_labels(sequences[chain])):
            digits = "".join(c for c in label if c.isdigit())
            if not digits:
                continue
            number = int(digits)
            imgt_label_of[(chain, index + 1)] = label
            imgt_region_of[(chain, index + 1)] = next(
                (name for name, (low, high) in IMGT_CDR_RANGES.items()
                 if low <= number <= high),
                "FR",
            )
    print(f"IMGT labels on {len(imgt_region_of)} positions "
          f"({sum(1 for v in imgt_region_of.values() if v != 'FR')} in a CDR)")
else:
    print("ANARCI is absent, so no IMGT regions: the position-matched cell will say so")

# --- distance to the antigen, and the area it buries ------------------------
# Both are geometry, computed here so the classes below can be selected without
# reference to any scorer. `distance_to_antigen` is the minimum heavy-atom
# distance from each antibody residue to any antigen heavy atom. `buried_area`
# is the side-chain area the antigen takes away: SASA of the residue in the
# antibody alone minus its SASA in the complex, the standard dSASA on complex
# formation. That is what "this residue is at the interface" means physically,
# as opposed to "FoldX says alanine costs something here".
complex_atoms = atoms[np.isin(atoms.chain_id,
                              [HEAVY_CHAIN, LIGHT_CHAIN, ANTIGEN_CHAIN])]
complex_sasa = struc.apply_residue_wise(
    complex_atoms, struc.sasa(complex_atoms, vdw_radii="Single"), np.nansum
)
complex_starts = struc.get_residue_starts(complex_atoms)
complex_chain = complex_atoms.chain_id[complex_starts]

buried_area = {}
for chain in (HEAVY_CHAIN, LIGHT_CHAIN):
    picked = np.flatnonzero(complex_chain == chain)
    if len(picked) != len(sequences[chain]):
        raise IndexMismatchError(
            f"chain {chain}: {len(picked)} residues in the complex SASA pass "
            f"against {len(sequences[chain])} in chain_sequence"
        )
    alone = np.flatnonzero(residue_chain == chain)
    for position, (here, there) in enumerate(zip(alone, picked), start=1):
        buried_area[(chain, position)] = (
            float(residue_sasa[here]) - float(complex_sasa[there])
        )

antigen_heavy = atoms[(atoms.chain_id == ANTIGEN_CHAIN) & (atoms.element != "H")]
is_heavy = antibody.element != "H"
distance_to_antigen = {}
# `bounds` holds ATOM indices and `residue_at` is keyed by RESIDUE index - the
# distinction the comment above exists for. So enumerate the residues and key on
# the counter, never on `begin`.
for residue, (begin, end) in enumerate(zip(bounds, bounds[1:])):
    keep = np.flatnonzero(is_heavy[begin:end]) + begin
    site = residue_at[residue]
    if not len(keep) or not antigen_heavy.array_length():
        distance_to_antigen[site] = float("inf")
        continue
    gaps = np.linalg.norm(
        antibody.coord[keep][:, None, :] - antigen_heavy.coord[None, :, :], axis=-1
    )
    distance_to_antigen[site] = float(gaps.min())

near = [k for k, v in distance_to_antigen.items()
        if v <= MIRROR_DISTANCE and in_variable_domain(*k)]
far = [k for k, v in distance_to_antigen.items()
       if v > NEARMISS_DISTANCE and in_variable_domain(*k)]
print()
print(f"distance to {ANTIGEN_CHAIN}: {len(near)} variable-domain positions within "
      f"{MIRROR_DISTANCE} A, {len(far)} beyond {NEARMISS_DISTANCE} A")
deepest = sorted(((v, k) for k, v in buried_area.items() if in_variable_domain(*k)),
                 reverse=True)[:12]
print(f"side-chain area buried by {ANTIGEN_CHAIN} (dSASA), the deepest {len(deepest)}:")
for area, (chain, position) in deepest:
    print(f"  {chain}:{sequences[chain][position - 1]}{position:<5}{area:>7.1f} A^2"
          f"   {distance_to_antigen[(chain, position)]:>5.1f} A to {ANTIGEN_CHAIN}")

# --- salt bridges, carboxyl oxygen to basic nitrogen ------------------------
ACID = {"ASP": ("OD1", "OD2"), "GLU": ("OE1", "OE2")}
BASE = {"LYS": ("NZ",), "ARG": ("NE", "NH1", "NH2"), "HIS": ("ND1", "NE2")}


def charged_atoms(table):
    names = np.array([str(n) for n in antibody.res_name])
    atom_names = np.array([str(n) for n in antibody.atom_name])
    keep = np.zeros(antibody.array_length(), dtype=bool)
    for residue, wanted in table.items():
        keep |= (names == residue) & np.isin(atom_names, list(wanted))
    return np.flatnonzero(keep)


acid_atoms, base_atoms = charged_atoms(ACID), charged_atoms(BASE)
bridges = set()
if len(acid_atoms) and len(base_atoms):
    gaps = np.linalg.norm(
        antibody.coord[acid_atoms][:, None, :] - antibody.coord[base_atoms][None, :, :],
        axis=-1,
    )
    for i, j in zip(*np.nonzero(gaps <= SALT_BRIDGE_CUTOFF)):
        acid_site = residue_at[int(residue_of_atom[acid_atoms[i]])]
        base_site = residue_at[int(residue_of_atom[base_atoms[j]])]
        if acid_site == base_site:
            continue
        bridges.add((acid_site, base_site))
in_fv = {pair for pair in bridges
         if in_variable_domain(*pair[0]) and in_variable_domain(*pair[1])}
print(f"\nsalt bridges within {SALT_BRIDGE_CUTOFF} A: {len(bridges)} acid-base "
      f"residue pairs over both whole chains, {len(in_fv)} of them with both "
      f"partners in the variable domains")
bridges = in_fv
print(f"  those {len(bridges)} pairs involve "
      f"{len({r for pair in bridges for r in pair})} residues")

paratope at 5.0 A: 21 positions (10 on B, 11 on A)

chains B and A: 434 residues, 227 of them in the variable domains (VH 1-120, VL 1-107)
secondary structure over the variable domains (biotite P-SEA): c=136, b=91  (a = helix, b = strand, c = coil)
relative SASA in the variable domains: 100 positions under 0.15 (buried), 51 over 0.4 (exposed)
IMGT labels on 227 positions (47 in a CDR)



distance to C: 21 variable-domain positions within 5.0 A, 165 beyond 10.0 A
side-chain area buried by C (dSASA), the deepest 12:
  A:N30      73.5 A^2     3.0 A to C
  A:Y92      72.9 A^2     3.0 A to C
  B:Y105     67.1 A^2     3.3 A to C
  B:Y57      66.2 A^2     3.5 A to C
  B:G103     65.8 A^2     2.4 A to C
  A:F53      64.2 A^2     3.6 A to C
  B:R59      55.0 A^2     3.4 A to C
  B:R50      29.1 A^2     2.6 A to C
  B:Y33      27.6 A^2     3.4 A to C
  A:H91      25.8 A^2     2.9 A to C
  A:T31      25.1 A^2     3.0 A to C
  A:T94      24.0 A^2     2.8 A to C

salt bridges within 4.0 A: 16 acid-base residue pairs over both whole chains, 10 of them with both partners in the variable domains
  those 10 pairs involve 17 residues


In [20]:
# --- build the manufactured classes -------------------------------------------------
# One variant per row, with the class it belongs to, what it is expected to do
# and which scorers may be graded on it. A position that qualifies for two
# classes is kept in the first and the overlap is printed, so no manufactured
# variant is counted twice.
rng = np.random.default_rng(DECOY_SEED)
FOLDX_DERIVED = "the FoldX alanine scan"


def capped(items, limit=None):
    '''A seeded sample, so re-running the notebook builds the same set.'''
    limit = MAX_PER_CLASS if limit is None else limit
    items = sorted(items)
    if len(items) <= limit:
        return items
    picked = rng.choice(len(items), size=limit, replace=False)
    return [items[i] for i in sorted(picked)]


proposals = []


# How each class was selected, which is what decides who may be graded on it.
# A class selected by an energy function cannot grade that function; a class
# selected from the coordinates can grade everything.
GEOMETRIC = "the coordinates (dSASA, distance, burial)"
CHEMICAL = "residue chemistry and secondary structure"


def propose(chain, position, mut, group, expect, excludes, why, criterion=CHEMICAL):
    wt = sequences[chain][position - 1]
    if wt == mut:
        return
    proposals.append({
        "group": group, "expect": expect, "excludes": excludes,
        "criterion": criterion,
        "chain": chain, "position": position, "wt": wt, "mut": mut,
        "label": f"{chain}:{wt}{position}{mut}", "why": why,
        "rel_sasa": relative_sasa.get((chain, position), float("nan")),
        "sse": sse_of.get((chain, position), "?"),
        "in_paratope": (chain, position) in paratope,
    })


# 1. Alanine at the ten FoldX hotspots.
hotspot_rows = [h for h in hotspots.HOTSPOTS
                if h.status == "hotspot" and h.ddg_bind is not None
                and h.ddg_bind >= HOTSPOT_DDG]
print(f"{len(hotspot_rows)} hotspots at or above {HOTSPOT_DDG} kcal/mol on alanine:")
for h in hotspot_rows:
    print(f"  {h.crystal_id:<8}{h.domain:<5}ddg_bind {h.ddg_bind:>5.2f} "
          f"({h.runs} runs)")
    if sequences[h.chain][h.position - 1] != h.wt:
        raise IndexMismatchError(
            f"hotspots says {h.crystal_id} but 1N8Z has "
            f"{sequences[h.chain][h.position - 1]} at {h.chain}:{h.position}"
        )
    propose(h.chain, h.position, "A", "hotspot_alanine", "break", FOLDX_DERIVED,
            f"alanine costs {h.ddg_bind:.2f} kcal/mol of binding (FoldX scan)")

# 2a. Proline in the middle of a beta strand: both neighbours also strand, so
# this is not an edge position where a proline is tolerated.
mid_strand = [
    (chain, position)
    for (chain, position), code in sorted(sse_of.items())
    if code == "b"
    and sse_of.get((chain, position - 1)) == "b"
    and sse_of.get((chain, position + 1)) == "b"
    and sequences[chain][position - 1] not in "PG"
    and in_variable_domain(chain, position)
]
for chain, position in capped(mid_strand):
    propose(chain, position, "P", "proline_in_strand", "break", None,
            "mid-strand: both neighbours are strand, so the hydrogen-bond ladder breaks")

# 2b. Glycine replacing a buried bulky side chain.
buried_bulky = [
    (chain, position)
    for (chain, position), value in sorted(relative_sasa.items())
    if value < BURIED_BELOW
    and sequences[chain][position - 1] in BULKY
    and in_variable_domain(chain, position)
]
for chain, position in capped(buried_bulky):
    propose(chain, position, "G", "glycine_for_buried_bulky", "break", None,
            f"relative SASA {relative_sasa[(chain, position)]:.3f}, a packed core position")

# 2c. Charge reversal at a salt bridge: acid to lysine, base to aspartate.
reversals = []
for (acid, base) in sorted(bridges):
    reversals.append((acid, "K", f"salt bridge with {base[0]}:{base[1]}"))
    reversals.append((base, "D", f"salt bridge with {acid[0]}:{acid[1]}"))
seen = set()
unique_reversals = []
for (chain, position), mut, why in reversals:
    if (chain, position) in seen:
        continue
    seen.add((chain, position))
    unique_reversals.append(((chain, position), mut, why))
for (chain, position), mut, why in capped(
    [(k, m, w) for k, m, w in unique_reversals]
):
    propose(chain, position, mut, "charge_reversal", "break", None,
            f"charge reversal, {why}")

# 3. Neutral decoys: alanine at exposed positions outside the paratope.
decoy_sites = [
    (chain, position)
    for (chain, position), value in sorted(relative_sasa.items())
    if value > SURFACE_ABOVE
    and (chain, position) not in paratope
    and sequences[chain][position - 1] != "A"
    and in_variable_domain(chain, position)
]
for chain, position in capped(decoy_sites, DECOY_POSITIONS):
    propose(chain, position, "A", "surface_decoy", "neutral", FOLDX_DERIVED,
            f"exposed (relative SASA {relative_sasa[(chain, position)]:.3f}) and "
            f"outside the paratope")

# 4. surface_nearmiss - the control on the controls, and not a break.
# The measured improvements replaced acids at *buried interface* positions. These
# are acids as far from the antigen as the domain allows and well exposed, given
# the same six substitutions. Nothing about the chemistry differs; only the
# structural context does. So a scorer with no structural input has no way to
# tell these from the real improvements and must land near 0.5 on the AUROC
# below - that is this class's own control, and a high score there means the
# class is confounded, not that the scorer is good.
nearmiss_sites = [
    (chain, position)
    for (chain, position), gap in sorted(distance_to_antigen.items())
    if gap > NEARMISS_DISTANCE
    and relative_sasa.get((chain, position), 0.0) > NEARMISS_RSA
    and sequences[chain][position - 1] in NEARMISS_ACIDS
    and in_variable_domain(chain, position)
]
print(f"{len(nearmiss_sites)} exposed acidic positions beyond "
      f"{NEARMISS_DISTANCE} A of {ANTIGEN_CHAIN}, x {len(NEARMISS_SUBS)} "
      f"substitutions = {len(nearmiss_sites) * len(NEARMISS_SUBS)} variants")
for chain, position in nearmiss_sites:
    for mut in NEARMISS_SUBS:
        propose(chain, position, mut, "surface_nearmiss", "neutral", None,
                f"acid {relative_sasa[(chain, position)]:.2f} exposed and "
                f"{distance_to_antigen[(chain, position)]:.1f} A from "
                f"{ANTIGEN_CHAIN}: the improvements' chemistry, none of their context",
                GEOMETRIC)

# 5. mirror - the improvements run backwards.
# K, R, F, W and Y are the classes that improved binding when installed at D31,
# D102 and D28. These are interface positions that already carry one of them,
# replaced by an acid. If a scorer liked the forward substitution because it
# understood the interface, it should dislike this; if it liked the forward one
# because it likes arginine, it will like this too.
mirror_sites = [
    (chain, position)
    for (chain, position), gap in sorted(distance_to_antigen.items())
    if gap <= MIRROR_DISTANCE
    and sequences[chain][position - 1] in MIRROR_WT
    and in_variable_domain(chain, position)
]
print(f"{len(mirror_sites)} interface positions within {MIRROR_DISTANCE} A "
      f"carrying {'/'.join(MIRROR_WT)}, x {len(MIRROR_SUBS)} = "
      f"{len(mirror_sites) * len(MIRROR_SUBS)} variants")
for chain, position in mirror_sites:
    for mut in MIRROR_SUBS:
        propose(chain, position, mut, "mirror", "break", None,
                f"{sequences[chain][position - 1]} at "
                f"{distance_to_antigen[(chain, position)]:.1f} A from "
                f"{ANTIGEN_CHAIN} replaced by an acid: the improvements reversed",
                GEOMETRIC)

# 6. geometric_hotspot_alanine - hotspots from the coordinates, so FoldX counts.
# The existing hotspot_alanine class came out of the FoldX scan, which is why
# FoldX is excluded from it. Buried area on complex formation is a geometric
# measurement that no scorer here produced, so every scorer including FoldX is
# graded on this one. The overlap with the FoldX list is printed below: where
# the two agree, the scan is cross-validated; where they disagree, one of them
# is worth a look.
by_area = sorted(((area, site) for site, area in buried_area.items()
                  if in_variable_domain(*site)), reverse=True)
geometric_sites = [site for area, site in by_area if area > DSASA_ABOVE]
if len(geometric_sites) < DSASA_TOP:
    geometric_sites = [site for _area, site in by_area[:DSASA_TOP]]
print(f"{len(geometric_sites)} positions burying more than {DSASA_ABOVE} A^2 "
      f"(or the top {DSASA_TOP}), alanine at each")
for chain, position in sorted(geometric_sites):
    propose(chain, position, "A", "geometric_hotspot_alanine", "break", None,
            f"buries {buried_area[(chain, position)]:.0f} A^2 on complex "
            f"formation, {distance_to_antigen[(chain, position)]:.1f} A from "
            f"{ANTIGEN_CHAIN}",
            GEOMETRIC)

# What the two hotspot definitions agree about.
foldx_sites = {(h.chain, h.position) for h in hotspot_rows}
geometric = set(geometric_sites)
both = sorted(foldx_sites & geometric)
print()
print(f"hotspot definitions: {len(foldx_sites)} from the FoldX scan, "
      f"{len(geometric)} from dSASA, {len(both)} in both")
for chain, position in sorted(foldx_sites | geometric):
    where = ("both" if (chain, position) in both
             else "FoldX only" if (chain, position) in foldx_sites else "dSASA only")
    print(f"  {chain}:{sequences[chain][position - 1]}{position:<5}"
          f"{buried_area.get((chain, position), float('nan')):>7.1f} A^2  {where}")
print("A position the scan calls a hotspot and the geometry does not is one FoldX")
print("thinks is energetically important without much buried area - plausible for")
print("a charged contact, worth checking for anything else.")
print()

manufactured = pd.DataFrame(proposals)
# The two hotspot classes are deliberately allowed to name the same
# substitution: they are the same alanine selected two different ways, and the
# overlap between them is the cross-validation printed above. Deduplicating
# across them would hand every shared position to whichever class came first -
# and the FoldX-derived one is exactly the class FoldX may not be graded on, so
# the geometric class would keep only its non-overlapping positions and FoldX
# would gain almost nothing. Every other pair of classes still dedups, so no
# variant is counted twice anywhere else, and the pooled rates dedup on
# variant_id so a shared alanine counts once there too.
HOTSPOT_CLASSES = ("hotspot_alanine", "geometric_hotspot_alanine")
shared_class = manufactured["group"].isin(HOTSPOT_CLASSES)
overlaps = manufactured[
    manufactured.duplicated(["chain", "position", "mut"], keep=False) & ~shared_class
]
if len(overlaps):
    print(f"\n{overlaps['label'].nunique()} substitution(s) proposed by more than one "
          f"class; kept in the first and dropped from the rest:")
    print(overlaps[["label", "group", "why"]].to_string(index=False))
manufactured = pd.concat([
    manufactured[~shared_class].drop_duplicates(["chain", "position", "mut"],
                                                keep="first"),
    manufactured[shared_class].drop_duplicates(["chain", "position", "mut", "group"],
                                               keep="first"),
], ignore_index=True)
hot_rows = manufactured[manufactured["group"].isin(HOTSPOT_CLASSES)]
both_labels = sorted(hot_rows.loc[hot_rows.duplicated("label", keep=False), "label"].unique())
if both_labels:
    print(f"{len(both_labels)} alanine(s) are in both hotspot classes, which is the "
          f"agreement between the two definitions:")
    print("  " + ", ".join(both_labels))
    print("FoldX is graded on the geometric copy and excused the scan's copy, so read")
    print("its geometric rate knowing many of those positions are ones its own scan")
    print("also picked. The dSASA-only rows are the independent part of that test.")
manufactured["variant_id"] = [
    make_id([(r.chain, r.position, r.wt, r.mut)]) for r in manufactured.itertuples()
]

# None of these may coincide with a measured variant: a manufactured negative
# that is actually in the panel is a measurement, and the two must not be mixed.
measured = set(direction["variant_id"]) | set(chinery["variant_id"])
clash = manufactured[manufactured["variant_id"].isin(measured)]
if len(clash):
    print(f"\n{len(clash)} manufactured variant(s) are also measured somewhere in the "
          f"benchmark and are dropped from this part:")
    print(clash[["label", "group", "why"]].to_string(index=False))
    manufactured = manufactured[~manufactured["variant_id"].isin(measured)]
manufactured = manufactured.reset_index(drop=True)

print(f"\n{len(manufactured)} manufactured variants:")
summary = (manufactured.groupby(["group", "expect", "criterion"])
           .agg(n=("label", "size"),
                in_paratope=("in_paratope", "sum"),
                median_rel_sasa=("rel_sasa", "median"))
           .reset_index())
print(summary.to_string(index=False))
print(f"\n'excludes' is which scorer may not be graded on a class because the class")
print(f"came out of that scorer:")
for group, excludes in manufactured.groupby("group")["excludes"].first().items():
    print(f"  {group:<28}{excludes if isinstance(excludes, str) else '(nothing)'}")
print(manufactured[["label", "group", "expect", "criterion", "sse", "rel_sasa",
                    "in_paratope", "why"]].to_string(index=False))

10 hotspots at or above 1.0 kcal/mol on alanine:
  B:Y33   VH   ddg_bind  2.04 (10 runs)
  B:R50   VH   ddg_bind  1.98 (10 runs)
  B:Y57   VH   ddg_bind  1.36 (10 runs)
  B:W99   VH   ddg_bind  1.97 (3 runs)
  B:Y105  VH   ddg_bind  1.13 (10 runs)
  A:N30   VL   ddg_bind  2.30 (3 runs)
  A:T31   VL   ddg_bind  1.34 (10 runs)
  A:F53   VL   ddg_bind  1.09 (10 runs)
  A:H91   VL   ddg_bind  1.29 (10 runs)
  A:T94   VL   ddg_bind  1.64 (10 runs)
4 exposed acidic positions beyond 10.0 A of C, x 6 substitutions = 24 variants
11 interface positions within 5.0 A carrying K/R/F/W/Y, x 2 = 22 variants
10 positions burying more than 40.0 A^2 (or the top 10), alanine at each

hotspot definitions: 10 from the FoldX scan, 10 from dSASA, 7 in both
  A:N30      73.5 A^2  both
  A:T31      25.1 A^2  FoldX only
  A:F53      64.2 A^2  both
  A:H91      25.8 A^2  both
  A:Y92      72.9 A^2  dSASA only
  A:T94      24.0 A^2  FoldX only
  B:Y33      27.6 A^2  both
  B:R50      29.1 A^2  both
  B:Y57      6

In [21]:
# --- score the manufactured set ---------------------------------------------
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="anarci is not installed")
    manufactured_table = enumerate_variants(
        complex_, HEAVY_CHAIN, out_dir / "manufactured" / "variants",
        only=manufactured["variant_id"].tolist(), structure=pdb,
    )
print(f"{manufactured_table.rows} rows, {manufactured_table.candidates} candidates")

print("\nscoring the manufactured set:")
manufactured_scored, manufactured_ran, manufactured_skipped = score_set(
    manufactured_table, ALL_MODELS, out_dir / "manufactured" / "scores", "manufactured set"
)
manufactured_panel = manufactured.merge(
    manufactured_scored.frame.drop_duplicates("variant_id")[
        ["variant_id"] + [f"score_{m}" for m in manufactured_ran]
    ],
    on="variant_id", how="left",
)

# Every manufactured variant is a single substitution, so every scorer can
# answer every one of them - which is the point: this is the one part of the
# notebook where the matrix scorers are tested on their full set.
if (manufactured_panel["position"] <= 0).any():
    raise ValueError("a manufactured variant has a non-positive position")
print(f"\nall {len(manufactured_panel)} manufactured variants are single substitutions, "
      f"so no scorer is excused by the lookup rule here.")

# The reference the harder criterion is measured against: the median score of
# the real positives from Part 1, per scorer.
positive_median = {}
for model in manufactured_ran:
    column = f"score_{model}"
    if column not in scored_frame.columns:
        positive_median[model] = float("nan")
        continue
    values = scored_frame.loc[scored_frame["direction"] == 1, column].to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    positive_median[model] = float(np.median(values)) if len(values) else float("nan")
print(f"\nthe median score of Part 1's measured improvers, per scorer, which is the")
print(f"bar the second criterion uses:")
for model, value in positive_median.items():
    n = int(np.isfinite(scored_frame.get(f'score_{model}', pd.Series(dtype=float))
                        .where(scored_frame['direction'] == 1)).sum())
    print(f"  {model:<14}{('-' if value != value else f'{value:+.3f}'):>10}  "
          f"over {n} scored improvers")

135 rows, 135 candidates

scoring the manufactured set:
  antifold         135 of 135 candidates scored
  ablang2          135 of 135 candidates scored


  esm1v            135 of 135 candidates scored
  esm2_650m        135 of 135 candidates scored
  antiberty        135 of 135 candidates scored
  consensus      SKIPPED  MissingDependencyError: the consensus scorer needs an OAS frequency table and none is configured. Pass frequencies=<path>, or set DESIGN_PIPELINE_OAS_TABLE. The file is a CSV of imgt,aa,frequency (or position,aa,frequency), built once from the Observed Antibody Space download at https://opig.stats.ox.ac.uk/webapps/oas/. Drop 'consensus' from models=[...] to run without it.
  antifold_seq     135 of 135 candidates scored
  foldx            135 of 135 candidates scored
  gearbind       SKIPPED  MissingRuntimeError: gearbind is not installed. Verified route (checked 2026-10-30) [container]: build an image from https://github.com/DeepGraphLearning/GearBind and set it with REGISTRY['gearbind'] - no published image exists. Note: TorchDrug pins Python 3.7-3.10 and an old torch: the one genuine conflict in this set, hence a co

In [22]:
# --- the failure rates ------------------------------------------------------
# Two criteria per class. `calls improving` is the share scored above
# FAILURE_ABOVE, which is the wild type by construction. `beats positives` is
# the share scored above the median of Part 1's measured improvers, which is
# what a shortlist cut would act on. The second is the one that matters and the
# first is the one that is unambiguous.
# The per-class table below reads every row; the pooled rates dedup on
# variant_id, because the two hotspot classes share alanines on purpose and a
# pooled failure rate has to count each substitution once.
breaks = (manufactured_panel[manufactured_panel["expect"] == "break"]
          .drop_duplicates("variant_id"))
decoys = (manufactured_panel[manufactured_panel["expect"] == "neutral"]
          .drop_duplicates("variant_id"))

rows = []
groups = list(manufactured_panel.groupby("group", sort=True).groups)
head = f"{'scorer':<14}{'class':<28}{'n':>4}{'calls improving':>17}{'beats positives':>17}{'median score':>14}"
print(head)
print("-" * len(head))
for model in manufactured_ran:
    column = f"score_{model}"
    for group in groups:
        subset = manufactured_panel[manufactured_panel["group"] == group]
        excludes = subset["excludes"].iloc[0]
        if isinstance(excludes, str) and model == "foldx":
            print(f"{model:<14}{group:<28}{'-':>4}{'not scored':>17}"
                  f"{'(circular)':>17}{'-':>14}")
            continue
        values = subset[column].to_numpy(dtype=float)
        usable = values[np.isfinite(values)]
        if not len(usable):
            print(f"{model:<14}{group:<28}{0:>4}{'-':>17}{'-':>17}{'-':>14}")
            continue
        improving = float((usable > FAILURE_ABOVE).mean())
        bar = positive_median[model]
        beats = float((usable > bar).mean()) if bar == bar else float("nan")
        print(f"{model:<14}{group:<28}{len(usable):>4}{improving:>16.0%}"
              f"{('-' if beats != beats else f'{beats:.0%}'):>17}"
              f"{np.median(usable):>+14.3f}")
        rows.append({"model": model, "group": group, "expect": subset["expect"].iloc[0],
                     "n": len(usable), "calls_improving": improving,
                     "beats_positives": beats, "median": float(np.median(usable))})
    print("-" * len(head))
failures = pd.DataFrame(rows)

# The headline per scorer: the failure rate over the deliberate breaks only,
# with the decoys reported separately because their expectation is different.
print(f"\n{'scorer':<14}{'breaks n':>10}{'failure rate':>14}{'beats positives':>17}"
      f"{'decoys n':>10}{'decoy median':>14}{'decoys > 0':>12}")
print("-" * 91)
manufactured_summary = []
for model in manufactured_ran:
    column = f"score_{model}"
    allowed = breaks[~((breaks["excludes"].notna()) & (model == "foldx"))]
    values = allowed[column].to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    bar = positive_median[model]
    rate = float((values > FAILURE_ABOVE).mean()) if len(values) else float("nan")
    beats = float((values > bar).mean()) if len(values) and bar == bar else float("nan")
    decoy_allowed = decoys[~((decoys["excludes"].notna()) & (model == "foldx"))]
    decoy_values = decoy_allowed[column].to_numpy(dtype=float)
    decoy_values = decoy_values[np.isfinite(decoy_values)]
    decoy_median = float(np.median(decoy_values)) if len(decoy_values) else float("nan")
    decoy_above = float((decoy_values > FAILURE_ABOVE).mean()) if len(decoy_values) else float("nan")
    print(f"{model:<14}{len(values):>10}"
          f"{('-' if rate != rate else f'{rate:.0%}'):>14}"
          f"{('-' if beats != beats else f'{beats:.0%}'):>17}"
          f"{len(decoy_values):>10}"
          f"{('-' if decoy_median != decoy_median else f'{decoy_median:+.3f}'):>14}"
          f"{('-' if decoy_above != decoy_above else f'{decoy_above:.0%}'):>12}")
    manufactured_summary.append({
        "model": model, "breaks_n": len(values), "failure_rate": rate,
        "breaks_beat_positives": beats, "decoys_n": len(decoy_values),
        "decoy_median": decoy_median, "decoys_above_zero": decoy_above,
    })
manufactured_summary = pd.DataFrame(manufactured_summary).set_index("model")
print("-" * 91)
print(f"'failure rate' is the share of deliberate breaks the scorer puts above the")
print(f"wild type. Read it against diagnostics.PLAUSIBLE_BENEFICIAL = "
      f"{diagnostics.PLAUSIBLE_BENEFICIAL:.0%}, which is")
print("this repo's loose upper bound on what share of substitutions to a matured")
print("antibody can plausibly be improvements - and these are not plausible ones.")
print()
print("'decoy median' should be near zero and 'decoys > 0' near half. A scorer with")
print("a large positive decoy median is enthusiastic about substitutions outside the")
print("paratope, which means its enthusiasm for a real design is about something")
print("other than the interface.")

path = out_dir / "manufactured" / "failures.csv"
path.parent.mkdir(parents=True, exist_ok=True)
failures.to_csv(path, index=False)
print(f"\n-> {path}")

scorer        class                          n  calls improving  beats positives  median score
----------------------------------------------------------------------------------------------
antifold      charge_reversal               16              0%               6%       -11.628
antifold      geometric_hotspot_alanine     10             10%              60%        -3.617
antifold      glycine_for_buried_bulky      20              0%               0%       -14.572
antifold      hotspot_alanine               10             10%              40%        -4.723
antifold      mirror                        22              9%              45%        -4.105
antifold      proline_in_strand             20              0%               0%       -16.230
antifold      surface_decoy                 20              0%              10%        -6.475
antifold      surface_nearmiss              24              4%              25%        -8.457
----------------------------------------------------------

In [23]:
from design_pipeline import REGISTRY

# --- the nearmiss comparison, matched on position ---------------------------
# The between-position version of this test does not work on this molecule, and
# the reason is worth printing rather than hiding: among the variable domains'
# acidic positions, CDR membership and distance to the antigen are perfectly
# collinear. Every acid in a CDR (A:D28 at 4.5 A, B:D31 9.6, B:D102 5.1,
# B:D108 9.3) is close to HER2; all thirteen acids beyond 10 A are framework.
# So "exposed acid far from the antigen" and "framework position" name the same
# residues here, and a sequence-only model separating the two groups has only
# told us it can tell a CDR from a framework - which it can, trivially. That is
# what the first version of this cell measured: esm1v 0.875, antiberty 0.806,
# ablang2 0.708, intervals all clear of 0.5, none of them given a coordinate.
acid_sites = sorted({
    (chain, position)
    for (chain, position), gap in distance_to_antigen.items()
    if in_variable_domain(chain, position)
    and sequences[chain][position - 1] in NEARMISS_ACIDS
})
print(f"{len(acid_sites)} acidic positions in the variable domains:")
print(f"{'region':<8}{'acids':>7}{'within ' + str(MIRROR_DISTANCE) + ' A':>14}"
      f"{'beyond ' + str(NEARMISS_DISTANCE) + ' A':>16}")
print("-" * 45)
collinear = True
for where in ("CDR1", "CDR2", "CDR3", "FR"):
    here = [s for s in acid_sites if imgt_region_of.get(s, "FR") == where]
    near = [s for s in here if distance_to_antigen[s] <= MIRROR_DISTANCE]
    far = [s for s in here if distance_to_antigen[s] > NEARMISS_DISTANCE]
    if near and far:
        collinear = False
    print(f"{where:<8}{len(here):>7}{len(near):>14}{len(far):>16}")
print("-" * 45)
if collinear:
    print("No region has acids on both sides of the distance split, so there is no")
    print("between-position comparison here that is not also a CDR-against-framework")
    print("comparison. The AUROC that used to be in this cell is therefore not")
    print("reported: it cannot be matched, and an unmatchable control is not a result.")
print()

# What CAN be matched is the position itself. At each position the measured
# improvements sit at, ask whether the scorer ranks those improvements above the
# OTHER substitutions at the SAME position. Same residue, same neighbours, same
# CDR, same burial - the only thing that varies is which amino acid goes in,
# which is the question a design shortlist actually asks.
#
# The weaker part of this label, stated plainly: the negatives are substitutions
# nobody measured, not substitutions measured to be bad. Some may be
# improvements that were never tried. That biases every AUROC below DOWNWARD and
# equally for every scorer, so the comparison between scorers is fair and the
# absolute value is a floor rather than an estimate.
reference_by_site = {}
for variant in NEARMISS_REFERENCE:
    chain, rest = variant.split(":")
    reference_by_site.setdefault((chain, int(rest[1:-1])), set()).add(rest[-1])
print(f"{len(reference_by_site)} positions carry the {len(NEARMISS_REFERENCE)} "
      f"measured improvements:")
for (chain, position), muts in sorted(reference_by_site.items()):
    wt = sequences[chain][position - 1]
    if wt != NEARMISS_REFERENCE[0].split(":")[1][0] and wt not in NEARMISS_ACIDS:
        raise IndexMismatchError(
            f"{chain}:{position} holds {wt}, but the reference calls it an acid"
        )
    print(f"  {chain}:{wt}{position:<5}"
          f"{imgt_region_of.get((chain, position), 'FR'):<6}"
          f"improvements to {''.join(sorted(muts))}, "
          f"{len(AMINO_ACIDS) - 1 - len(muts)} other substitutions as negatives")

# The negative pool is "substitutions nobody measured", and that has to be
# true. Several substitutions at these positions DO carry a measurement
# elsewhere in the benchmark as a design of their own, and leaving them in the
# pool labels a measured variant unmeasured. Derived from the file rather than
# listed by hand, so it stays right if the benchmark grows.
#
# The rule is a single-substitution measurement: a design that is exactly this
# substitution and nothing else. A substitution appearing inside a nine-way
# CDR-H3 redesign is not a measurement of that substitution, and excluding on
# that basis would empty the pool - 17 of the 19 at D102 appear in some Chinery
# design. What these carry is a Chinery label, which Part 3 showed is clean
# kinetics and not affinity, so they are REMOVED rather than promoted to
# positives: measured, but not for this question.
measured_singles = {}
for row in negatives_file.itertuples():
    if int(row.n_mutations) != 1:
        continue
    chain = CHAIN_MAP.get(str(row.chain), str(row.chain))
    measured_singles.setdefault((chain, int(row.position)), {}).setdefault(
        str(row.mut), set()
    ).add(str(row.label_class))

matched_rows, excluded_rows = [], []
for (chain, position), muts in sorted(reference_by_site.items()):
    wt = sequences[chain][position - 1]
    here = measured_singles.get((chain, position), {})
    for mut in AMINO_ACIDS:
        if mut == wt:
            continue
        row = {
            "variant_id": make_id([(chain, position, wt, mut)]),
            "site": f"{chain}:{wt}{position}", "chain": chain, "position": position,
            "wt": wt, "mut": mut, "improved": int(mut in muts),
        }
        if mut not in muts and mut in here:
            row["measured_as"] = ",".join(sorted(here[mut]))
            excluded_rows.append(row)
            continue
        matched_rows.append(row)
matched = pd.DataFrame(matched_rows).drop_duplicates("variant_id")

if excluded_rows:
    excluded = pd.DataFrame(excluded_rows)
    print()
    print(f"{len(excluded)} substitution(s) dropped from the negative pool because the "
          f"benchmark measures them on their own:")
    for row in excluded.itertuples():
        print(f"  {row.site}{row.mut:<4}measured as {row.measured_as}")
    print("A Chinery `positive` is a clean 1:1 fit, not an improvement (Part 3), so")
    print("these are removed from the negatives rather than counted as positives.")
    excluded.to_csv(out_dir / "manufactured" / "matched_excluded.csv", index=False)
print()
print(f"{len(matched)} substitutions over {matched['site'].nunique()} positions, "
      f"{int(matched['improved'].sum())} of them measured improvements")
for site, group in matched.groupby("site"):
    print(f"  {site:<9}{int(group['improved'].sum())} improvements, "
          f"{int((group['improved'] == 0).sum())} unmeasured negatives")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="anarci is not installed")
    matched_table = enumerate_variants(
        complex_, HEAVY_CHAIN, out_dir / "manufactured" / "matched_variants",
        only=matched["variant_id"].tolist(), structure=pdb,
    )
print()
print("scoring the position-matched set:")
matched_scored, matched_ran, _matched_skipped = score_set(
    matched_table, ALL_MODELS, out_dir / "manufactured" / "matched_scores",
    "position-matched set"
)
matched_panel = matched.merge(
    matched_scored.frame.drop_duplicates("variant_id")[
        ["variant_id"] + [f"score_{m}" for m in matched_ran]
    ],
    on="variant_id", how="left",
)


def stratified_auroc(frame, column, stratum="site", label="improved"):
    """Concordance pooled within strata: every pair compared at one position.

    Concordant pairs over comparable pairs, ties at a half. Not the mean of
    per-stratum AUROCs, which would weight a position carrying two improvements
    the same as one carrying four.
    """
    wins = pairs = 0.0
    for _site, group in frame.groupby(stratum):
        values = group[column].to_numpy(dtype=float)
        flags = group[label].to_numpy(dtype=int)
        good = values[np.isfinite(values) & (flags == 1)]
        rest = values[np.isfinite(values) & (flags == 0)]
        if not len(good) or not len(rest):
            continue
        gaps = good[:, None] - rest[None, :]
        wins += float((gaps > 0).sum()) + 0.5 * float((gaps == 0).sum())
        pairs += float(len(good) * len(rest))
    return (wins / pairs if pairs else float("nan")), pairs


def stratified_ci(frame, column, resamples=None):
    """Percentile CI, resampling the POSITIVES within each position.

    The unit of independent information here is the measured improvement, and
    there are nine of them. The 142 pairs are nine positives crossed with the
    negatives at the same position, so resampling pairs - or resampling both
    sides - treats one measurement as if it were sixteen and returns an
    interval far narrower than the evidence supports. The negatives are a large
    fixed reference pool and are held fixed; what is resampled is the nine.

    A position contributing one positive contributes one draw, so a bootstrap
    sample can duplicate or drop whole improvements, which is exactly the
    variability that matters at this n.
    """
    resamples = BOOTSTRAP if resamples is None else resamples
    groups = [group for _site, group in frame.groupby("site")]
    draws = []
    for _ in range(int(resamples)):
        parts = []
        for group in groups:
            good = group[group["improved"] == 1]
            rest = group[group["improved"] == 0]
            if not len(good) or not len(rest):
                continue
            parts.append(pd.concat([
                good.sample(len(good), replace=True,
                            random_state=int(RNG.integers(1 << 32))),
                rest,
            ]))
        if not parts:
            return float("nan"), float("nan")
        point, _pairs = stratified_auroc(pd.concat(parts), column)
        if point == point:
            draws.append(point)
    if not draws:
        return float("nan"), float("nan")
    return (float(np.percentile(draws, 100 * (1 - CI) / 2)),
            float(np.percentile(draws, 100 * (1 + CI) / 2)))


head = (f"{'scorer':<14}{'structure':>11}{'pairs':>7}{'AUROC':>8}"
        f"{'  ' + str(int(CI * 100)) + '% CI':>16}   per position")
print()
print(head)
print("-" * (len(head) + 18))
matched_out = []
for model in matched_ran:
    column = f"score_{model}"
    adapter = REGISTRY.get(model)
    structural = getattr(adapter, "requires_structure", True) if adapter else True
    point, pairs = stratified_auroc(matched_panel, column)
    low, high = stratified_ci(matched_panel, column, resamples=MATCHED_BOOTSTRAP)
    per_site = []
    for site, group in matched_panel.groupby("site"):
        one, _p = stratified_auroc(group, column)
        per_site.append(f"{site} {'-' if one != one else format(one, '.2f')}")
    interval = "-" if low != low else f"{low:.3f} to {high:.3f}"
    print(f"{model:<14}{('yes' if structural else 'no'):>11}{int(pairs):>7}"
          f"{cell_text(point):>8}{interval:>16}   {'  '.join(per_site)}")
    matched_out.append({"model": model, "requires_structure": structural,
                        "pairs": int(pairs), "auroc": point,
                        "ci_low": low, "ci_high": high})
print("-" * (len(head) + 18))
matched_auroc = pd.DataFrame(matched_out).set_index("model")
matched_path = out_dir / "manufactured" / "position_matched_auroc.csv"
matched_auroc.to_csv(matched_path)
print(f"-> {matched_path}")
print()
print("The interval resamples the nine measured improvements, not the 142 pairs:")
print("one measurement is one unit of evidence however many negatives it is paired")
print("against, and a pair-level bootstrap would report an interval several times")
print("too narrow.")
print("This one is matched: every pair is two substitutions at the same position of")
print("the same molecule, so nothing here is won by recognising a CDR. A scorer at")
print("0.5 cannot tell which substitution at a paratope acid was the one that")
print("worked, which is the decision a shortlist makes.")


17 acidic positions in the variable domains:
region    acids  within 5.0 A   beyond 10.0 A
---------------------------------------------
CDR1          2             1               0
CDR2          0             0               0
CDR3          2             0               0
FR           13             0              13
---------------------------------------------
No region has acids on both sides of the distance split, so there is no
between-position comparison here that is not also a CDR-against-framework
comparison. The AUROC that used to be in this cell is therefore not
reported: it cannot be matched, and an unmatchable control is not a result.

3 positions carry the 9 measured improvements:
  A:D28   CDR1  improvements to KR, 17 other substitutions as negatives
  B:D31   CDR1  improvements to KMR, 16 other substitutions as negatives
  B:D102  CDR3  improvements to FHKM, 15 other substitutions as negatives



4 substitution(s) dropped from the negative pool because the benchmark measures them on their own:
  B:D102E   measured as positive
  B:D102G   measured as positive
  B:D102N   measured as positive
  B:D102Q   measured as positive
A Chinery `positive` is a clean 1:1 fit, not an improvement (Part 3), so
these are removed from the negatives rather than counted as positives.

53 substitutions over 3 positions, 9 of them measured improvements
  A:D28    2 improvements, 17 unmeasured negatives
  B:D102   4 improvements, 11 unmeasured negatives
  B:D31    3 improvements, 16 unmeasured negatives



scoring the position-matched set:
  antifold          53 of 53 candidates scored
  ablang2           53 of 53 candidates scored
  esm1v             53 of 53 candidates scored
  esm2_650m         53 of 53 candidates scored
  antiberty         53 of 53 candidates scored


  consensus      SKIPPED  MissingDependencyError: the consensus scorer needs an OAS frequency table and none is configured. Pass frequencies=<path>, or set DESIGN_PIPELINE_OAS_TABLE. The file is a CSV of imgt,aa,frequency (or position,aa,frequency), built once from the Observed Antibody Space download at https://opig.stats.ox.ac.uk/webapps/oas/. Drop 'consensus' from models=[...] to run without it.
  antifold_seq      53 of 53 candidates scored
  foldx             53 of 53 candidates scored
  gearbind       SKIPPED  MissingRuntimeError: gearbind is not installed. Verified route (checked 2026-10-30) [container]: build an image from https://github.com/DeepGraphLearning/GearBind and set it with REGISTRY['gearbind'] - no published image exists. Note: TorchDrug pins Python 3.7-3.10 and an old torch: the one genuine conflict in this set, hence a container rather than Env B.. `python -m design_pipeline.preflight` prints the whole table.
position-matched set: scorers that ran: antifold, ablang

antifold              yes    126   0.349  0.262 to 0.429   A:D28 0.41  B:D102 0.16  B:D31 0.48


ablang2                no    126   0.246  0.143 to 0.333   A:D28 0.32  B:D102 0.07  B:D31 0.35


esm1v                  no    126   0.333  0.198 to 0.437   A:D28 0.44  B:D102 0.18  B:D31 0.40


esm2_650m              no    126   0.262  0.167 to 0.341   A:D28 0.41  B:D102 0.14  B:D31 0.27


antiberty              no    126   0.317  0.167 to 0.468   A:D28 0.44  B:D102 0.11  B:D31 0.42


antifold_seq          yes    126   0.278  0.127 to 0.437   A:D28 0.32  B:D102 0.27  B:D31 0.25


foldx                 yes    126   0.810  0.714 to 0.889   A:D28 0.94  B:D102 0.55  B:D31 0.96
-----------------------------------------------------------------------------------------
-> runs/negative_controls/manufactured/position_matched_auroc.csv

The interval resamples the nine measured improvements, not the 142 pairs:
one measurement is one unit of evidence however many negatives it is paired
against, and a pair-level bootstrap would report an interval several times
too narrow.
This one is matched: every pair is two substitutions at the same position of
the same molecule, so nothing here is won by recognising a CDR. A scorer at
0.5 cannot tell which substitution at a paratope acid was the one that
worked, which is the decision a shortlist makes.


In [24]:
# --- is the matched result just conservativeness? ---------------------------
# A mechanism worth ruling in or out. Every scorer below ranked the measured
# improvements at these three acids BELOW the substitutions nobody measured, and
# the obvious explanation is exchangeability: a masked language model trained on
# natural sequence puts its mass on residues that normally appear in place of an
# aspartate, and the improvements are not those residues. If that is what is
# happening, two things follow and both are checkable here:
#
#   1. each scorer's within-position ranking correlates with BLOSUM62 from the
#      wild type, and
#   2. BLOSUM62 used as a scorer reproduces their AUROC on this same test.
#
# The second is the decisive one. A correlation with exchangeability is only an
# explanation of the failure if exchangeability itself fails the test.
try:
    from Bio.Align import substitution_matrices
except ModuleNotFoundError:
    substitution_matrices = None
    print("biopython is not installed, so this cell is skipped: "
          "`uv pip install biopython`. Nothing else depends on it.")

if substitution_matrices is not None:
    from scipy.stats import spearmanr

    blosum = substitution_matrices.load("BLOSUM62")
    wild_types = sorted({row.wt for row in matched_panel.itertuples()})
    print(f"BLOSUM62 from the wild type at these positions ({', '.join(wild_types)}):")
    for wt in wild_types:
        ordered = sorted(((float(blosum[wt, m]), m) for m in AMINO_ACIDS if m != wt),
                         reverse=True)
        print("  " + "  ".join(f"{wt}->{m}{int(s):+d}" for s, m in ordered))

    matched_panel["blosum"] = [
        float(blosum[row.wt, row.mut]) for row in matched_panel.itertuples()
    ]
    improvements = matched_panel[matched_panel["improved"] == 1]
    print()
    print("the measured improvements, and how exchangeable they are:")
    for site, group in improvements.groupby("site"):
        pairs = "  ".join(f"{row.wt}->{row.mut}{int(row.blosum):+d}"
                          for row in group.itertuples())
        print(f"  {site:<9}{pairs}")
    print("None of them is a conservative choice, but none is the least conservative")
    print("either: they sit mid-table, which is already a hint about what follows.")

    # 2 first, because it decides what 1 means.
    blosum_point, blosum_pairs = stratified_auroc(matched_panel, "blosum")
    blosum_low, blosum_high = stratified_ci(matched_panel, "blosum",
                                            resamples=MATCHED_BOOTSTRAP)
    print()
    print(f"BLOSUM62 AS A SCORER on the position-matched test: "
          f"{cell_text(blosum_point)} "
          f"({blosum_low:.3f} to {blosum_high:.3f}) over {int(blosum_pairs)} pairs")
    for site, group in matched_panel.groupby("site"):
        one, _pairs = stratified_auroc(group, "blosum")
        print(f"  {site:<9}{cell_text(one)}")

    # 1, now interpretable.
    head = (f"{'scorer':<14}{'AUROC':>8}{'rho vs BLOSUM':>15}   per position")
    print()
    print(head)
    print("-" * (len(head) + 14))
    conservative_rows = []
    for model in matched_ran:
        column = f"score_{model}"
        usable = matched_panel[np.isfinite(matched_panel[column])]
        pooled = (float(spearmanr(usable[column], usable["blosum"]).statistic)
                  if len(usable) > 2 else float("nan"))
        per_site = []
        for site, group in usable.groupby("site"):
            rho = (float(spearmanr(group[column], group["blosum"]).statistic)
                   if len(group) > 2 else float("nan"))
            per_site.append(f"{site} {'-' if rho != rho else format(rho, '+.2f')}")
        point, _pairs = stratified_auroc(matched_panel, column)
        print(f"{model:<14}{cell_text(point):>8}"
              f"{('-' if pooled != pooled else format(pooled, '+.3f')):>15}   "
              f"{'  '.join(per_site)}")
        conservative_rows.append({"model": model, "matched_auroc": point,
                                  "rho_vs_blosum": pooled})
    print("-" * (len(head) + 14))
    conservativeness = pd.DataFrame(conservative_rows).set_index("model")
    conservativeness.loc["blosum62"] = [blosum_point, 1.0]
    blosum_path = out_dir / "manufactured" / "conservativeness.csv"
    conservativeness.to_csv(blosum_path)
    print(f"-> {blosum_path}")

    # The reading, written as a conditional on the numbers rather than after them.
    print()
    explains = blosum_high < 0.5
    if explains:
        print("BLOSUM62 alone fails this test, so exchangeability is sufficient to")
        print("produce the anti-correlation and the scorers that track it inherit the")
        print("failure. That is a mechanism, not just a negative result.")
    else:
        print("BLOSUM62 alone is at chance here, so exchangeability does NOT explain")
        print("the anti-correlation. The improvements are mid-table substitutions, not")
        print("conservative ones, so a scorer ranking by exchangeability lands near")
        print("0.5 - and the scorers below are well under it. Whatever they are")
        print("tracking costs them more than conservativeness would, and the partial")
        print("correlations in the table are not the explanation. The hypothesis is")
        print("checkable and checked, and it is not the answer.")


BLOSUM62 from the wild type at these positions (D):
  D->E+2  D->N+1  D->S+0  D->Q+0  D->T-1  D->P-1  D->K-1  D->H-1  D->G-1  D->R-2  D->A-2  D->Y-3  D->V-3  D->M-3  D->I-3  D->F-3  D->C-3  D->W-4  D->L-4

the measured improvements, and how exchangeable they are:
  A:D28    D->K-1  D->R-2
  B:D102   D->F-3  D->H-1  D->K-1  D->M-3
  B:D31    D->K-1  D->M-3  D->R-2
None of them is a conservative choice, but none is the least conservative
either: they sit mid-table, which is already a hint about what follows.



BLOSUM62 AS A SCORER on the position-matched test: 0.536 (0.433 to 0.639) over 126 pairs
  A:D28    0.574
  B:D102   0.591
  B:D31    0.458

scorer           AUROC  rho vs BLOSUM   per position
------------------------------------------------------------------
antifold         0.349         +0.177   A:D28 +0.51  B:D102 -0.11  B:D31 +0.62
ablang2          0.246         +0.079   A:D28 -0.41  B:D102 +0.06  B:D31 +0.45
esm1v            0.333         +0.343   A:D28 +0.44  B:D102 +0.23  B:D31 +0.74
esm2_650m        0.262         +0.425   A:D28 +0.71  B:D102 +0.19  B:D31 +0.69
antiberty        0.317         +0.169   A:D28 +0.44  B:D102 +0.07  B:D31 +0.61
antifold_seq     0.278         +0.066   A:D28 +0.50  B:D102 -0.09  B:D31 +0.30
foldx            0.810         -0.206   A:D28 -0.14  B:D102 -0.09  B:D31 -0.29
------------------------------------------------------------------
-> runs/negative_controls/manufactured/conservativeness.csv

BLOSUM62 alone is at chance here, so exchangeability does

### `wt_argmax_rate` - every interface position as its own control

The cheapest control in the set, and the only one that needs no negative at
all. Trastuzumab is a matured, clinically optimised antibody. At a paratope
position, a model that understands this molecule should mostly prefer **the
residue that is already there** over the other nineteen.

`design_pipeline.metrics.diagnostics.wt_argmax_rate` measures exactly that, and
the docstring is blunt about what a low rate means: it does not weaken the run,
it invalidates it - the model is not reading this antibody, and every number in
the column is then about something else. The threshold to read it against is
`EXPECTED_WT_ARGMAX`, 0.6.

Two mechanical points:

* **It needs the full alphabet at a position**, so this is a separate matrix run
  over the paratope only: nineteen substitutions at each interface position.
  `enumerate_variants` skips wild-type-to-wild-type because it is not a variant,
  so the wild type's own row is added here with a score of **exactly zero** -
  which is not a convenience, it is the definition. Every matrix scorer in this
  repo reports `M[position, mut] - M[position, wt]`, so the wild type's score
  against itself is zero by construction and not by assumption.
* **Matrix scorers only.** On 1N8Z at 5 A the paratope is 21 positions, so the
  full alphabet is 399 candidates - one forward pass for `antifold`, one pass
  per position for the language models, and nothing for the rest.
  `antifold_seq`, `foldx` and `gearbind` are one pass *per candidate*, so the
  same table would be 399 runs of a thing that takes seconds to minutes each.
  The question is also not quite theirs: they condition on the whole mutant, so
  "the best residue at this position" is not a quantity they compute, it is one
  you would have to assemble from 399 separate answers.

In [25]:
# --- the paratope, the full alphabet, one matrix run ------------------------
interface_ids = []
for chain, position in sorted(paratope):
    wt = sequences[chain][position - 1]
    if wt == "X":
        continue
    for mut in AMINO_ACIDS:
        if mut == wt:
            continue
        interface_ids.append(make_id([(chain, position, wt, mut)]))
print(f"{len(sorted(paratope))} paratope positions x 19 substitutions = "
      f"{len(interface_ids)} candidates")

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="anarci is not installed")
    interface_table = enumerate_variants(
        complex_, HEAVY_CHAIN, out_dir / "manufactured" / "interface_variants",
        only=interface_ids, structure=pdb,
    )
print(f"{interface_table.rows} rows, {interface_table.candidates} candidates")

print("\nscoring the paratope alphabet (matrix scorers only):")
interface_scored, interface_ran, interface_skipped = score_set(
    interface_table, MATRIX_MODELS, out_dir / "manufactured" / "interface_scores",
    "paratope alphabet"
)

# The wild-type rows, at the score zero means by construction. Not a fill and
# not an imputation: every matrix scorer here reports M[position, mut] minus
# M[position, wt], so the wild type against itself is exactly 0.0.
wt_rows = pd.DataFrame([
    {"variant_id": make_id([(chain, position, sequences[chain][position - 1],
                             sequences[chain][position - 1])]),
     "chain": chain, "position": position,
     "wt": sequences[chain][position - 1], "mut": sequences[chain][position - 1]}
    for chain, position in sorted(paratope)
    if sequences[chain][position - 1] != "X"
])
for model in interface_ran:
    wt_rows[f"score_{model}"] = 0.0

columns = ["variant_id", "chain", "position", "wt", "mut"] + \
          [f"score_{m}" for m in interface_ran]
with_wt = pd.concat(
    [interface_scored.frame.reindex(columns=[c for c in columns
                                             if c in interface_scored.frame.columns]),
     wt_rows.reindex(columns=columns)],
    ignore_index=True,
)
print(f"\n{len(with_wt)} rows: {len(interface_scored.frame)} scored substitutions plus "
      f"{len(wt_rows)} wild-type rows at score 0.0 (the definition, not a fill)")

print(f"\n{'scorer':<14}{'positions':>11}{'scored':>8}{'skipped':>9}{'wt argmax rate':>16}"
      f"{'  verdict':>12}")
print("-" * 72)
wt_argmax = []
for model in interface_ran:
    result = diagnostics.wt_argmax_rate(with_wt, f"score_{model}")
    rate = result["rate"]
    verdict = ("-" if rate != rate
               else "ok" if rate >= diagnostics.EXPECTED_WT_ARGMAX else "INVALID")
    print(f"{model:<14}{result['positions']:>11}{result['scored']:>8}"
          f"{result['skipped']:>9}{('-' if rate != rate else f'{rate:.1%}'):>16}"
          f"{verdict:>12}")
    wt_argmax.append({"model": model, "positions": result["positions"],
                      "scored": result["scored"], "wt_argmax_rate": rate})
wt_argmax = pd.DataFrame(wt_argmax).set_index("model")
print("-" * 72)
print(f"read against diagnostics.EXPECTED_WT_ARGMAX = "
      f"{diagnostics.EXPECTED_WT_ARGMAX:.0%}. Materially below it and the column is")
print("not about this antibody, whatever its AUROC in Parts 2 and 3 came out at.")
print()
print("One honest complication: at a paratope position the residue that is there")
print("was selected for binding HER2, and AntiFold with no antigen in its input")
print("is conditioned on the Fab alone - so it is being asked to prefer a residue")
print("chosen for a reason it cannot see. A rate below the threshold is therefore")
print("weaker evidence of a bug for the structure-based scorers than for the")
print("sequence-based ones, which see the matured sequence itself.")

21 paratope positions x 19 substitutions = 399 candidates
399 rows, 399 candidates

scoring the paratope alphabet (matrix scorers only):
  antifold         399 of 399 candidates scored
  ablang2          399 of 399 candidates scored
  esm1v            399 of 399 candidates scored
  esm2_650m        399 of 399 candidates scored
  antiberty        399 of 399 candidates scored
  consensus      SKIPPED  MissingDependencyError: the consensus scorer needs an OAS frequency table and none is configured. Pass frequencies=<path>, or set DESIGN_PIPELINE_OAS_TABLE. The file is a CSV of imgt,aa,frequency (or position,aa,frequency), built once from the Observed Antibody Space download at https://opig.stats.ox.ac.uk/webapps/oas/. Drop 'consensus' from models=[...] to run without it.
paratope alphabet: scorers that ran: antifold, ablang2, esm1v, esm2_650m, antiberty



420 rows: 399 scored substitutions plus 21 wild-type rows at score 0.0 (the definition, not a fill)

scorer          positions  scored  skipped  wt argmax rate     verdict
------------------------------------------------------------------------
antifold               21      21        0           66.7%          ok
ablang2                21      21        0            9.5%     INVALID
esm1v                  21      21        0           14.3%     INVALID
esm2_650m              21      21        0           23.8%     INVALID


antiberty              21      21        0           19.0%     INVALID
------------------------------------------------------------------------
read against diagnostics.EXPECTED_WT_ARGMAX = 60%. Materially below it and the column is
not about this antibody, whatever its AUROC in Parts 2 and 3 came out at.

One honest complication: at a paratope position the residue that is there
was selected for binding HER2, and AntiFold with no antigen in its input
is conditioned on the Fab alone - so it is being asked to prefer a residue
chosen for a reason it cannot see. A rate below the threshold is therefore
weaker evidence of a bug for the structure-based scorers than for the
sequence-based ones, which see the matured sequence itself.


## Part 5 - the verdict table

One row per scorer. Four columns of evidence and one verdict, and the verdict is
a conjunction of stated criteria rather than a judgement:

| criterion | threshold | why that one |
|---|---|---|
| direction AUROC | the lower end of the bootstrap CI above 0.5 | the point estimate on 21 against 8 is not evidence; the interval is the claim |
| Chinery AUROC | the lower end of the bootstrap CI above 0.5 | the same rule on the panel large enough for the interval to be narrow |
| manufactured failure rate | at or below `PLAUSIBLE_BENEFICIAL` (25%) | this repo's existing bound on what share of substitutions can plausibly be improvements, and these are not plausible ones |
| `wt_argmax_rate` | at or above `EXPECTED_WT_ARGMAX` (60%), matrix scorers only | below it, the column is not about this antibody |

A scorer passes only if every criterion that applies to it passes. A criterion
that could not be measured - no route on this box, too few of one label - is
reported as not measured and the verdict is `unproven`, which is not `fail`:
nothing was shown either way.

**And the multiple-comparisons point, since Part 5 is where a winner gets
picked.** Several scorers are tested against the same small panels. With `k`
scorers the chance that at least one clears 0.5 by luck grows with `k`, and no
correction is applied below because applying one to 21-against-8 would be
decorating a number that cannot carry it. Instead the count is printed beside
the result: **whichever scorer wins, it won against this many others on this
much data**, and that is the sentence that should travel with it.

In [26]:
# --- the verdict, one column per claim --------------------------------------
# Two panels, two different labels, two different n. A single conjunctive
# verdict let the Chinery panel's 253 wells lend their power to an affinity
# claim that rests on 29 measured variants, and let a thin direction panel veto
# a well-powered kinetics result. So each claim is judged on its own evidence
# and printed in its own column, and there is no combined "pass" at all.
AFFINITY_CRITERIA = ("direction",)       # Part 2: 21 improved against 8 worsened
KINETICS_CRITERIA = ("chinery",)         # Part 3: 156 fitted against 97 I.C.
# Floor checks. Not a third claim: a column that calls a deliberate break an
# improvement, or that cannot find the matured residue, is unusable for either
# claim, so these apply to both.
FLOOR_CRITERIA = ("manufactured", "wt_argmax")

rows = []
for model in ALL_MODELS:
    row = {"model": model,
           "kind": "matrix" if model in MATRIX_MODELS else "multi"}
    if model in overall.index:
        row["direction_auroc"] = overall.loc[model, "auroc"]
        row["direction_ci_low"] = overall.loc[model, "ci_low"]
        row["direction_ci_high"] = overall.loc[model, "ci_high"]
        row["direction_p"] = overall.loc[model, "p"]
        row["direction_n"] = int(overall.loc[model, "n_pos"] + overall.loc[model, "n_neg"])
    else:
        row.update({"direction_auroc": float("nan"), "direction_ci_low": float("nan"),
                    "direction_ci_high": float("nan"), "direction_p": float("nan"),
                    "direction_n": 0})
    if model in chinery_clean_auroc.index:
        row["chinery_auroc"] = chinery_clean_auroc.loc[model, "auroc"]
        row["chinery_ci_low"] = chinery_clean_auroc.loc[model, "ci_low"]
        row["chinery_ci_high"] = chinery_clean_auroc.loc[model, "ci_high"]
        row["chinery_n"] = int(chinery_clean_auroc.loc[model, "n_pos"]
                               + chinery_clean_auroc.loc[model, "n_neg"])
    else:
        row.update({"chinery_auroc": float("nan"), "chinery_ci_low": float("nan"),
                    "chinery_ci_high": float("nan"), "chinery_n": 0})
    if model in manufactured_summary.index:
        row["failure_rate"] = manufactured_summary.loc[model, "failure_rate"]
        row["decoy_median"] = manufactured_summary.loc[model, "decoy_median"]
    else:
        row.update({"failure_rate": float("nan"), "decoy_median": float("nan")})
    row["wt_argmax"] = (wt_argmax.loc[model, "wt_argmax_rate"]
                        if model in wt_argmax.index else float("nan"))
    rows.append(row)
verdict = pd.DataFrame(rows).set_index("model")

CRITERIA = {
    "direction": lambda row: (row["direction_ci_low"], lambda v: v > 0.5),
    "chinery": lambda row: (row["chinery_ci_low"], lambda v: v > 0.5),
    "manufactured": lambda row: (row["failure_rate"],
                                 lambda v: v <= diagnostics.PLAUSIBLE_BENEFICIAL),
    "wt_argmax": lambda row: (row["wt_argmax"],
                              lambda v: v >= diagnostics.EXPECTED_WT_ARGMAX),
}


def judge(row, criteria):
    '''(verdict, reasons) over `criteria` plus the floor checks that apply.'''
    failed, unmeasured = [], []
    applicable = list(criteria) + [
        name for name in FLOOR_CRITERIA
        if name != "wt_argmax" or row["kind"] == "matrix"
    ]
    for name in applicable:
        value, test = CRITERIA[name](row)
        if value != value:
            unmeasured.append(name)
        elif not test(value):
            failed.append(name)
    if failed:
        return "FAIL", "fails " + ", ".join(failed)
    if unmeasured:
        return "unproven", "not measured: " + ", ".join(unmeasured)
    return "pass", "every criterion met"


head = (f"{'scorer':<14}{'kind':>7}"
        f"{'AFFINITY: direction AUROC (CI)':>32}{'n':>5}{'  claim':<11}"
        f"{'KINETICS: Chinery AUROC (CI)':>31}{'n':>6}{'  claim':<11}"
        f"{'fail':>6}{'wt':>6}")
print(head)
print("-" * len(head))
for model, row in verdict.iterrows():
    affinity, affinity_why = judge(row, AFFINITY_CRITERIA)
    kinetics, kinetics_why = judge(row, KINETICS_CRITERIA)
    verdict.loc[model, "affinity_verdict"] = affinity
    verdict.loc[model, "affinity_reason"] = affinity_why
    verdict.loc[model, "kinetics_verdict"] = kinetics
    verdict.loc[model, "kinetics_reason"] = kinetics_why
    direction_cell = (
        "not scored" if row["direction_auroc"] != row["direction_auroc"]
        else f"{row['direction_auroc']:.3f} ({row['direction_ci_low']:.2f}-{row['direction_ci_high']:.2f})"
    )
    chinery_cell = (
        "not scored" if row["chinery_auroc"] != row["chinery_auroc"]
        else f"{row['chinery_auroc']:.3f} ({row['chinery_ci_low']:.2f}-{row['chinery_ci_high']:.2f})"
    )
    print(f"{model:<14}{row['kind']:>7}{direction_cell:>32}"
          f"{(row['direction_n'] or '-'):>5}  {affinity:<9}"
          f"{chinery_cell:>31}{(row['chinery_n'] or '-'):>6}  {kinetics:<9}"
          f"{('-' if row['failure_rate'] != row['failure_rate'] else f'{row.failure_rate:.0%}'):>6}"
          f"{('-' if row['wt_argmax'] != row['wt_argmax'] else f'{row.wt_argmax:.0%}'):>6}")
print("-" * len(head))
print("AFFINITY rests on the Part 2 direction set and nothing else. KINETICS rests")
print("on Part 3's clean/unfittable split, which Part 3 shows is a developability")
print("label and not an affinity one. A scorer can pass one and fail the other, and")
print("neither column is evidence for the other's claim. Both carry the floor checks")
print("(manufactured failure rate, and wt_argmax for the matrix scorers), because a")
print("column that fails those is not usable for any claim.")

for label, column, reason_column, panel in (
    ("AFFINITY (direction)", "affinity_verdict", "affinity_reason", len(direction)),
    ("CLEAN KINETICS (Chinery)", "kinetics_verdict", "kinetics_reason",
     int(chinery_clean_auroc[["n_pos", "n_neg"]].max().sum())
     if len(chinery_clean_auroc) else 0),
):
    passed = verdict.index[verdict[column] == "pass"].tolist()
    unproven = verdict.index[verdict[column] == "unproven"].tolist()
    measured = int(verdict[column].ne("unproven").sum())
    print(f"\n{label}, n = {panel}:")
    if passed:
        print(f"  pass: {', '.join(passed)} - against {max(measured - 1, 0)} other "
              f"scorer(s) on the same {panel} items. That sentence travels with the")
        print("  result: testing several scorers on one panel and reporting the best is")
        print("  how a null result becomes a claim.")
    else:
        print("  pass: none. Nothing here is shown usable for this claim, which is as")
        print("  much a statement about the panel's size as about the scorers.")
    if unproven:
        print(f"  unproven: {', '.join(unproven)} - something was not measurable, "
              f"which is not a failure.")
        for model in unproven:
            print(f"    {model:<14}{verdict.loc[model, reason_column]}")

path = out_dir / "verdict.csv"
verdict.to_csv(path)
print(f"\n-> {path}")

scorer           kind  AFFINITY: direction AUROC (CI)    n  claim       KINETICS: Chinery AUROC (CI)     n  claim      fail    wt
---------------------------------------------------------------------------------------------------------------------------------
antifold       matrix               0.700 (0.40-0.97)   13  FAIL                          not scored    14  unproven     3%   67%
ablang2        matrix               0.667 (0.00-1.00)   13  FAIL                          not scored    14  FAIL        30%   10%
esm1v          matrix               0.233 (0.00-0.60)   13  FAIL                          not scored    14  FAIL        11%   14%
esm2_650m      matrix               0.400 (0.00-0.80)   13  FAIL                          not scored    14  FAIL        18%   24%
antiberty      matrix               0.367 (0.00-0.77)   13  FAIL                          not scored    14  FAIL        29%   19%
consensus      matrix                      not scored    -  unproven                      

## What this notebook licenses, and what it does not

Fill the first paragraph in from the run. The rest is true before it runs,
because it is about the panels and not about the results.

**The claim the verdict table supports** is a negative one at the level of a
filter: a scorer whose direction interval contains 0.5, or whose manufactured
failure rate is above a quarter, should not be cutting a shortlist. That is
actionable without any positive claim about any scorer.

**The claim it does not support** is a ranking between scorers. Two scorers
whose intervals overlap are not ordered by this panel, and almost all of them
overlap.

### The caveats, restated so they travel with the numbers

* **`L:R66G` is a weak negative.** The source paper describes the affinity as
  *similar* and reports **4.8 +/- 0.7 nM against 2.0 +/- 0.4 nM with no
  significance test.** It is one of eight negatives and one of three singles, so
  it carries real weight in Part 2; the sensitivity table there reports every
  AUROC again with it dropped, and that delta is the honest measure of how much
  of the result rests on it.

* **The Chinery block is selected on outcome, so correlations there are
  attenuated.** 156 clean fits out of 720 submitted. Both the weakest and the
  unmeasurable candidates are gone, the dynamic range is truncated, and the
  truncation is size-dependent - a bigger design is likelier to fail to fit at
  all, so the large-size bins are the survivors of a harsher filter. Every AUROC
  in Part 3 is a lower bound, and the within-size columns on the right-hand side
  describe a different population from the ones on the left.

* **The `N/A` label is ambiguous in the source** and the evidence CSV refuses to
  call it a demonstrated non-binder. Part 3 reports the table with and without
  those 434 wells and neither as the answer. A scorer that looks better with
  them in may be separating "this well went wrong" from "this bound".

* **Several scorers were tested against small panels.** Part 5 prints how many,
  beside whichever wins. No multiplicity correction is applied, because
  correcting a 21-against-8 panel would be dressing up a number that cannot
  carry the weight; naming the count is the honest alternative.

* **Direction is pooled, magnitude is not.** Three formats - Fab, IgG, scFv -
  and three assays. The sign is comparable and nothing else is, which is why
  there is no KD, no fold change and no ddG anywhere in Parts 2 to 5.

* **The manufactured negatives are not a measurement.** They can show a scorer
  is broken. They cannot show one works, and a scorer that passes Part 4 has
  only been shown not to be obviously wrong. FoldX is excluded from the two
  alanine classes because the hotspot list is its own output; it is graded on the
  other three, which came from the coordinates.

### What would actually settle it

* **More measured negatives, from one format and one assay.** Eight negatives
  across three formats is the binding constraint on everything above, and the
  interval does not narrow until that number grows. Thirty Fab losses on one
  instrument against one batch-matched parent would do more than any modelling
  change in this repo.
* **A design set not filtered on whether it bound.** Even a coarse three-bin
  readout over all 720 Chinery designs would be worth more here than 156 clean
  KDs, because it is the 564 missing ones that carry the size dependence.
* **A positive set with repeats of one substitution in several contexts.**
  `H:D102W` is the best single in the panel, a component of an 8-fold
  improvement, and a component of a loss. One substitution measured in five
  backgrounds would turn that anecdote into a number, and that number is the
  ceiling on every additive scorer in the registry.